<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">
<br>
<h1 style="font-family:verdana; font-weight: bold; color: #5A4636; text-align: center;">
Deteksi Ketidaksesuaian Judul dan Isi Berita<br>
<span style="font-size: 22px;">End-to-End Notebook DAC IFEST 2026</span>
</h1>

<p style="text-align:center; font-size:16px; color:#6F5C4F;">
Tim <b>psi-1</b>
</p>

<p style="text-align:center; font-size:15px; color:#6F5C4F; line-height:1.7;">
Ketua <b>Muhammad Aufar Rizqi Kusuma - 13524061</b><br>
Anggota <b>Mochamad Fachri Alfaridzi - 13525128</b><br>
Anggota <b>Kurt Mikhael Purba - 13524065</b>
</p>


# Daftar Isi
1. [Pendahuluan](#1)
2. [Problem Statement dan Ketentuan Lomba](#2)
3. [Dataset Overview](#3)
4. [Initialization](#4)
5. [Exploratory Data Analysis](#5)
6. [Preprocessing](#6)
7. [Modelling](#7)
8. [Training Retraining dan Submission](#8)
9. [Reproducibility dan Audit Akhir](#9)
10. [Post Insight Analysis](#10)


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Pendahuluan <a name="1"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Judul adalah pintu masuk berita. Pembaca daring memindai judul terlebih dahulu lalu memutuskan untuk membuka isi atau tidak. Kebiasaan ini membuat judul memikul beban besar sebagai ringkasan yang jujur. Ketika judul mengklaim sesuatu yang tidak ada di isi pembaca mendapat gambaran yang keliru. Kekeliruan kecil pada isu sensitif dapat meluas menjadi kegaduhan di ruang publik.

Masalah ini terlihat sederhana tetapi sulit diotomatisasi. Di literatur masalah ini dikenal sebagai Headline Incongruity Detection. Cara paling sederhana adalah memeriksa apakah kata pada judul muncul di isi. Cara tersebut gagal ketika judul dan isi memakai kata yang sama untuk peristiwa yang berlawanan. Penghitungan skor kemiripan kata juga gagal pada parafrasa dan pada perbedaan kecil nama atau angka yang mengubah makna. Kajian pada berita berbahasa Indonesia menunjukkan pola kegagalan yang sama.

Notebook ini disusun sebagai solusi final Tim psi-1 untuk Data Analysis Competition IFEST 2026. Tujuan utamanya adalah membangun model klasifikasi yang menilai kesesuaian judul dan isi secara kontekstual bukan sekadar kemiripan kata. Seluruh pipeline dirancang agar dapat direproduksi dan diaudit dari hulu ke hilir mulai dari pembacaan data mentah sampai penulisan submission.

Batasan yang dijaga di notebook ini
- Model dilatih from scratch tanpa bobot pra latih apapun
- Tidak ada pemanggilan API eksternal atau LLM
- Pemilihan threshold dan bobot hanya dari validasi internal tanpa melihat label test
- Seed dikunci agar hasil dapat diulang


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Problem Statement dan Ketentuan Lomba <a name="2"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Problem Statement

Kompetisi meminta peserta membangun model klasifikasi biner untuk pasangan judul dan isi berita. Untuk setiap pasangan pada data uji model harus memprediksi salah satu dari dua label berikut

| Label | Arti |
|---|---|
| 1 Sesuai | Informasi utama pada judul didukung dan dibahas di isi |
| 0 Tidak Sesuai | Informasi utama pada judul tidak didukung oleh isi atau merujuk peristiwa berbeda |

Dataset berisi pasangan judul dan isi. Data latih dilengkapi label sedangkan data uji tidak berlabel dan harus diprediksi. Tantangan utamanya adalah kemiripan kata yang tinggi pada kedua kelas sehingga model yang hanya mengandalkan overlap akan salah.

Output akhir berupa file CSV dengan dua kolom yaitu id dan label yang mengikuti format sample_submission.csv.

---

## Ketentuan Lomba dan Evaluasi

Berdasarkan guidebook dan instruction resmi

- Task adalah klasifikasi biner judul dan isi
- Metrik evaluasi utama adalah Macro F1 Score yaitu rata rata F1 kelas 0 dan kelas 1. Metrik ini dipilih karena distribusi kelas tidak seimbang. Setiap kelas memberi kontribusi yang sama sehingga model tidak boleh hanya mengejar kelas dominan

$$\text{Macro F1} = \frac{\text{F1}_0 + \text{F1}_1}{2}$$

- Larangan pra latih berlaku penuh meliputi BERT GPT LLM ResNet VGG IndoBERT FastText pra latih dan embedding pra latih lainnya. Arsitektur yang sama tetap boleh dipakai bila bobot diinisialisasi acak dan dilatih sepenuhnya dari awal
- AutoML tidak diperkenankan kecuali komponen HPO seperti Optuna
- Data eksternal diperbolehkan bila publik gratis dan bukan ground truth serta wajib dicantumkan sumbernya di bagian atas notebook
- Pelabelan dan augmentasi hanya boleh pada data latih. Data uji hanya dipakai pada tahap inferensi
- Notebook wajib reproducible dengan seed


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Dataset Overview <a name="3"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Sumber Dataset

Dataset utama bersumber dari kompetisi resmi Kaggle IFEST 2026

- Link kompetisi ada pada laman Kaggle resmi DAC IFEST 2026
- File yang dipakai pada notebook ini adalah data/train.csv data/test.csv dan data/sample_submission.csv yang tersedia di folder data

Dataset eksternal yang dipakai pada notebook ini bersifat publik gratis dan bukan ground truth. Rincian lengkap ada pada tabel di bawah serta lisensi tersedia di data/external_data/README.md

| File | Sumber | Lisensi | Dipakai untuk |
|---|---|---|---|
| provinces.csv regencies.csv districts.csv | github.com/edwardsamuel/Wilayah-Administratif-Indonesia BPS 2018 | MIT untuk skrip dan ODbL untuk data | gazetteer wilayah dan hierarki provinsi |
| id-names.txt | github.com/perlancar/references | publik | leksikon nama orang |
| kbba.txt | github.com/insomniagung/kamus_kbba | publik | normalisasi slang ke baku |
| stopwords-id.txt | github.com/stopwords-iso/stopwords-id | MIT | daftar stopword untuk overlap tanpa stopword |
| tesaurus.json | github.com/victoriasovereigne/tesaurus Pusat Bahasa 2008 | publik | dukungan sinonim |

---

## Ringkasan Dataset Lomba

| Split | Jumlah baris | Kolom |
|---|---|---|
| Train | 14397 | id title content label |
| Test | 3603 | id title content |
| Sample submission | 3603 | id label |

Distribusi label pada train adalah 12960 Sesuai dan 1437 Tidak Sesuai atau sekitar 90 persen berbanding 10 persen. Rasio ini penting karena Macro F1 memberi bobot yang sama pada kelas kecil. Model yang sekadar memprediksi semua Sesuai akan mendapat Macro F1 sekitar 0.47 sehingga penanganan imbalance menjadi wajib.

Karakter tambahan yang memengaruhi desain pipeline
- Judul rata rata sekitar 10 kata isi rata rata sekitar 310 kata dengan pencilan isi sampai lebih dari 4000 kata
- Kemiripan kata judul ke isi sangat tinggi pada kedua kelas. Mean overlap label 1 sekitar 0.796 dan label 0 sekitar 0.749 pada dataset terbaru. Hard negative dengan overlap di atas 0.7 berjumlah 954 baris
- Topik Covid mendominasi judul sehingga model harus hati hati agar tidak menghafal kata kunci Covid saja

---

## Struktur Direktori

```text
data/
  train.csv
  test.csv
  sample_submission.csv
  external_data/
    provinces.csv regencies.csv districts.csv
    id-names.txt kbba.txt stopwords-id.txt tesaurus.json
outputs/
artifact/
  eda_figures/
```


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Initialization <a name="4"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Bagian ini menyiapkan fondasi sebelum pipeline berjalan

1. Pemeriksaan perangkat komputasi apakah CPU atau GPU
2. Penyiapan path agar notebook dapat berjalan di local Kaggle maupun Colab
3. Penguncian seed agar eksperimen dapat diulang dengan perilaku yang konsisten


## Pemeriksaan GPU dan Environment

Sel berikut memastikan perangkat GPU driver CUDA dan kompatibilitas environment. Notebook dirancang untuk berjalan di local Windows atau Linux dengan Python 3.10 ke atas serta di Kaggle Notebooks atau Google Colab yang menyediakan GPU T4 atau P100.

Jika Anda menjalankan di Kaggle pastikan dataset input sudah tersedia dan path akan disesuaikan otomatis. Di local pastikan folder data dan data/external_data sudah ada.


In [ ]:
import subprocess, sys
try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    print(out.stdout if out.stdout else "nvidia-smi tidak tersedia, lanjut dengan CPU")
    if out.stderr:
        print(out.stderr)
except Exception as e:
    print(f"cek GPU dilewati karena {type(e).__name__}: {e}")
    print("pipeline tetap berjalan di CPU")


## Path Setup dan Kompatibilitas Environment

Sel ini menetapkan root project dan path penting lain yang dipakai di seluruh pipeline. Notebook menggunakan mekanisme fallback agar otomatis menemukan folder yang berisi data/train.csv dan data/external_data.


In [ ]:
from pathlib import Path
import os, sys

def detect_environment():
    if 'KAGGLE_DATA_PROXY_URL' in os.environ or 'KAGGLE_USER_SECRETS_TOKEN' in os.environ:
        return 'kaggle'
    try:
        import google.colab  # noqa
        return 'colab'
    except Exception:
        return 'local'

ENV = detect_environment()
print(f"Environment terdeteksi {ENV}")

if ENV == 'kaggle':
    PROJECT_ROOT = Path("/kaggle/input/dac-ifest-2026")
    # fallback ke working dir bila input tidak sesuai
    if not (PROJECT_ROOT / "data").exists():
        PROJECT_ROOT = Path.cwd()
elif ENV == 'colab':
    PROJECT_ROOT = Path("/content/drive/MyDrive/IFEST-DAC")
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = Path("/content")
else:
    PROJECT_ROOT = Path.cwd()
    # cari ke atas bila dijalankan dari subfolder
    for cand in [PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent]:
        if (cand / "data" / "train.csv").exists():
            PROJECT_ROOT = cand
            break

DATA_DIR = PROJECT_ROOT / "data"
EXT_DIR = DATA_DIR / "external_data"
OUT_DIR = PROJECT_ROOT / "outputs"
ARTIFACT_DIR = PROJECT_ROOT / "artifact"
EDA_DIR = ARTIFACT_DIR / "eda_figures"

for p in [OUT_DIR, ARTIFACT_DIR, EDA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT {PROJECT_ROOT}")
print(f"DATA_DIR {DATA_DIR} exists {DATA_DIR.exists()}")
print(f"EXT_DIR {EXT_DIR} exists {EXT_DIR.exists()}")
print(f"OUT_DIR {OUT_DIR}")
print(f"EDA_DIR {EDA_DIR}")


## Imports dan Penguncian Seed

Sel berikut memuat pustaka inti dan mengunci seluruh sumber acak. Word2Vec dijalankan dengan workers sama dengan satu dan LightGBM dengan deterministic True agar perbedaan antar run sangat kecil. Iterasi pada himpunan yang memengaruhi hasil disortir secara eksplisit.


In [ ]:
import os
os.environ.setdefault("PYTHONHASHSEED", "0")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import random, re, time, warnings, json, collections
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics import roc_auc_score, f1_score, classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 150, "font.size": 9})
sns.set_style("whitegrid")

SEED = 42
N_THREADS = 8
N_SPLITS = 5
MODEL_SEEDS = (0, 1)

random.seed(SEED)
np.random.seed(SEED)
import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print(f"catatan determinisme {type(e).__name__}")

DEV = "cuda" if torch.cuda.is_available() else "cpu"

import sklearn, lightgbm, xgboost, catboost, gensim
print(f"python {sys.version.split()[0]}")
print(f"numpy {np.__version__} pandas {pd.__version__} sklearn {sklearn.__version__}")
print(f"lightgbm {lightgbm.__version__} xgboost {xgboost.__version__} catboost {catboost.__version__}")
print(f"gensim {gensim.__version__} torch {torch.__version__} device {DEV}")
print(f"SEED {SEED} threads {N_THREADS}")

T0 = time.time()
def tick(msg):
    print(f"[{time.time()-T0:7.1f}s] {msg}", flush=True)

tick("imports dan seed siap")


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Exploratory Data Analysis <a name="5"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

EDA pada notebook ini bukan pajangan visual. Setiap analisis dirancang untuk menjawab pertanyaan desain pipeline. Setelah setiap visual ada dua hal yaitu insight apa yang didapat dan dampaknya ke penyusunan pipeline akhir. Seluruh EDA dihitung from scratch tanpa pra latih dan kodenya adalah port dari eda.py agar konsisten dengan laporan.


## 1 Struktur dan Integritas Dataset

Sebelum masuk ke isi teks kita perlu memastikan bahwa file yang dipakai bersih. Pemahaman dimensi jumlah kolom keberadaan nilai hilang dan duplikasi menjadi fondasi sebelum analisis lanjutan.


In [ ]:
# muat data untuk EDA
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

# fallback untuk struktur lama _v1_backup bila ada
if not train_path.exists() and (PROJECT_ROOT / "data" / "_v1_backup" / "train.csv").exists():
    train_path = PROJECT_ROOT / "data" / "_v1_backup" / "train.csv"
    test_path = PROJECT_ROOT / "data" / "_v1_backup" / "test.csv"

train_eda = pd.read_csv(train_path)
test_eda = pd.read_csv(test_path)
sample_eda = pd.read_csv(sample_path) if sample_path.exists() else None

print(f"train shape {train_eda.shape} columns {list(train_eda.columns)}")
print(f"test shape {test_eda.shape} columns {list(test_eda.columns)}")
if sample_eda is not None:
    print(f"sample_submission shape {sample_eda.shape} columns {list(sample_eda.columns)}")
print()
print("dtypes train")
print(train_eda.dtypes)
print()
print("missing per kolom train")
print(train_eda.isnull().sum())
print("missing per kolom test")
print(test_eda.isnull().sum())
print()
print("duplikat id train", train_eda["id"].duplicated().sum())
print("duplikat id test", test_eda["id"].duplicated().sum())
print("duplikat exact title plus content pada train", train_eda.duplicated(subset=["title","content"]).sum())
print("duplikat id di sample submission", sample_eda["id"].duplicated().sum() if sample_eda is not None else "tidak ada file")

# simpan ringkasan untuk audit
(train_eda.head(3).to_string(), test_eda.head(3).to_string())


Insight yang didapat
Dataset bersih dan tidak ada nilai hilang pada kolom kunci. Tidak ada duplikat id baik di train maupun test yang berarti pemisahan train dan test rapi. Terdapat tiga baris dengan pasangan judul dan isi yang identik secara persis pada train. Sampel ini tidak mengganggu pemodelan tetapi perlu diperhatikan saat grouping karena pasangan identik dapat terbelah antar fold bila tidak dikelompokkan.

Dampak ke pipeline
Pipeline memakai grouping berbasis judul ternormalisasi dan isi ternormalisasi dengan union find agar baris yang berbagi judul atau isi berada pada fold yang sama. Langkah ini mencegah kebocoran validasi dan membuat estimasi OOF lebih jujur.


## 2 Distribusi Split Train dan Test

Kita perlu mengetahui proporsi data latih dan uji. Proporsi yang ekstrem atau perubahan distribusi antara train dan test akan memengaruhi strategi validasi dan kalibrasi threshold.


In [ ]:
import matplotlib.pyplot as plt

split_counts = pd.Series({"train": len(train_eda), "test": len(test_eda)})
print(split_counts)
fig, ax = plt.subplots(figsize=(5,3.2))
ax.bar(["train","test"], split_counts.values, color=["#8E7B6B","#C4A484"], edgecolor="white")
for i, v in enumerate(split_counts.values):
    ax.text(i, v+80, str(v), ha="center", va="bottom", fontsize=10)
ax.set_title("Distribusi Split Train dan Test")
ax.set_ylabel("Jumlah baris")
plt.tight_layout()
plt.savefig(EDA_DIR / "eda_split.png", dpi=150)
plt.show()


Insight yang didapat
Train berisi 14397 baris dan test berisi 3603 baris dengan rasio sekitar 80 banding 20. Rasio ini umum untuk kompetisi dan tidak menunjukkan anomali.

Dampak ke pipeline
Ukuran test yang cukup besar membuat kalibrasi threshold dari OOF menjadi krusial. Notebook tidak mengambil threshold dari test. Seluruh threshold dipilih dengan memaksimalkan Macro F1 pada OOF lalu diterapkan apa adanya ke test di ruang skor yang sama.


## 3 Distribusi Kelas dan Deteksi Ketidakseimbangan

Ketidakseimbangan kelas adalah isu utama pada metrik Macro F1. Kita perlu memeriksa seberapa timpang distribusi label karena hal ini menentukan pembobotan loss dan teknik sampling.


In [ ]:
vc = train_eda["label"].value_counts().sort_index()
print(vc)
print(f"proporsi label 0 {vc[0]/len(train_eda):.3f} label 1 {vc[1]/len(train_eda):.3f}")
print(f"rasio imbalance {vc.max()/vc.min():.2f} kali")

fig, ax = plt.subplots(figsize=(5,3.2))
ax.bar(["0 Tidak Sesuai","1 Sesuai"], vc.values, color=["#ef4444","#22c55e"], edgecolor="white")
for i, v in enumerate(vc.values):
    ax.text(i, v+150, f"{v}\n({v/len(train_eda):.1%})", ha="center", va="bottom", fontsize=9)
ax.set_title("Distribusi Label Train n sama dengan 14397 imbalance 90 banding 10")
ax.set_ylabel("Jumlah")
plt.tight_layout()
plt.savefig(EDA_DIR / "eda_label_dist.png", dpi=150)
plt.show()

# naive baseline
from sklearn.metrics import f1_score
import numpy as np
y_tmp = train_eda["label"].to_numpy()
pred_all1 = np.ones_like(y_tmp)
print(f"naive semua 1 Macro F1 {(f1_score(y_tmp, pred_all1, average='macro')):.4f}")


Insight yang didapat
Label 1 Sesuai mendominasi sekitar 90 persen sedangkan label 0 Tidak Sesuai hanya sekitar 10 persen. Jika model memprediksi semua Sesuai maka Macro F1 hanya sekitar 0.47. Hal ini menunjukkan bahwa akurasi tidak relevan dan model harus dioptimalkan khusus untuk kelas kecil.

Dampak ke pipeline
Pipeline memakai tiga penyeimbang sekaligus yaitu class weight balanced pada GBDT scale pos weight pada XGBoost dan penentuan threshold dari OOF yang memaksimalkan Macro F1 bukan akurasi. Pada cross encoder data negatif diperbanyak dengan pengulangan dan korupsi sintetis agar kelas kecil mendapat sinyal yang cukup.


## 4 Panjang Teks Judul dan Isi

Panjang teks memberi gambaran tentang risiko truncating dan apakah panjang itu sendiri sudah menjadi sinyal. Kita memeriksa panjang dalam karakter dan jumlah kata.


In [ ]:
for df in (train_eda, test_eda):
    df["title_len"] = df["title"].str.len()
    df["content_len"] = df["content"].str.len()
    df["title_wc"] = df["title"].str.split().str.len()
    df["content_wc"] = df["content"].str.split().str.len()

desc = train_eda[["title_len","title_wc","content_len","content_wc"]].describe().round(2)
print("train describe")
print(desc.to_string())
desc_test = test_eda[["title_len","title_wc","content_len","content_wc"]].describe().round(2)
print("\ntest describe")
print(desc_test.to_string())
by_label = train_eda.groupby("label")[["title_wc","content_wc","title_len","content_len"]].mean().round(1)
print("\nrata rata per label")
print(by_label.to_string())

# boxplot
import seaborn as sns
fig, axes = plt.subplots(1,2, figsize=(7,3.2), sharey=False)
sns.boxplot(data=train_eda, x="label", y="title_wc", ax=axes[0], palette=["#ef4444","#22c55e"], hue="label", legend=False)
axes[0].set_title("Judul jumlah kata")
sns.boxplot(data=train_eda, x="label", y="content_wc", ax=axes[1], palette=["#ef4444","#22c55e"], hue="label", legend=False)
axes[1].set_title("Isi jumlah kata")
plt.suptitle("Boxplot Panjang Teks per Label")
plt.tight_layout()
plt.savefig(EDA_DIR / "eda_boxplot_lengths.png", dpi=150)
plt.show()

# kde distribusi
for col, title in [("title_wc","Judul jumlah kata"), ("content_wc","Isi jumlah kata")]:
    fig, ax = plt.subplots(figsize=(6,3.2))
    for lab, color, label in [(1,"#22c55e","Sesuai 1"),(0,"#ef4444","Tidak Sesuai 0")]:
        sns.kdeplot(train_eda.loc[train_eda.label==lab, col], ax=ax, fill=True, alpha=0.25, color=color, label=label)
        sns.histplot(train_eda.loc[train_eda.label==lab, col], ax=ax, bins=40, alpha=0.15, color=color, stat="density")
    ax.set_title(f"Distribusi {title} per Label")
    ax.legend()
    plt.tight_layout()
    plt.savefig(EDA_DIR / f"eda_dist_{col}.png", dpi=150)
    plt.show()


Insight yang didapat
Judul rata rata sekitar 10 kata dan isi rata rata sekitar 310 kata. Median judul 10 dan median isi 276. Rentang isi sangat lebar dari 3 kata sampai lebih dari 4000 kata. Rata rata per label hampir identik sehingga panjang saja tidak membedakan kelas. Pencilan isi yang sangat panjang ada pada kedua kelas dan juga pada test.

Dampak ke pipeline
Panjang tidak dipakai sebagai fitur utama. Isi dipotong pada batas yang aman yaitu 1200 karakter pertama untuk TF IDF dan 200 sampai 400 token untuk cross encoder agar pencilan tidak mengacaukan representasi. Fitur panjang hanya dipakai sebagai fitur numerik pendukung bukan penentu.


## 5 Overlap Judul ke Isi dan Hard Subset

Overlap adalah fitur paling intuitif untuk tugas ini. Kita hitung Jaccard berbasis himpunan kata antara judul dan isi lalu lihat distribusinya per label serta subset sulit yaitu hard negative dan hard positive.


In [ ]:
def overlap_ratio(title, content):
    t = set(str(title).lower().split())
    c = set(str(content).lower().split())
    return len(t & c) / len(t) if t else 0

train_eda["overlap"] = [overlap_ratio(a,b) for a,b in zip(train_eda["title"], train_eda["content"])]
test_eda["overlap"] = [overlap_ratio(a,b) for a,b in zip(test_eda["title"], test_eda["content"])]

odesc = train_eda.groupby("label")["overlap"].describe().round(3)
print(odesc.to_string())
hard_neg = (train_eda.label==0) & (train_eda.overlap>0.7)
hard_pos = (train_eda.label==1) & (train_eda.overlap<0.5)
print(f"\nhard negative overlap di atas 0.7 {hard_neg.sum()} baris persentase terhadap label 0 {(hard_neg.sum()/(train_eda.label==0).sum()):.1%}")
print(f"hard positive overlap di bawah 0.5 {hard_pos.sum()} baris persentase terhadap label 1 {(hard_pos.sum()/(train_eda.label==1).sum()):.1%}")
print(f"mean overlap test {test_eda.overlap.mean():.3f} train {train_eda.overlap.mean():.3f}")

fig, ax = plt.subplots(figsize=(6,3.2))
sns.kdeplot(train_eda.loc[train_eda.label==1, "overlap"], ax=ax, fill=True, alpha=0.25, color="#22c55e", label="Sesuai 1")
sns.histplot(train_eda.loc[train_eda.label==1, "overlap"], ax=ax, bins=40, alpha=0.15, color="#22c55e", stat="density")
sns.kdeplot(train_eda.loc[train_eda.label==0, "overlap"], ax=ax, fill=True, alpha=0.25, color="#ef4444", label="Tidak Sesuai 0")
sns.histplot(train_eda.loc[train_eda.label==0, "overlap"], ax=ax, bins=40, alpha=0.15, color="#ef4444", stat="density")
ax.set_title("Distribusi Overlap Judul ke Isi per Label")
ax.legend()
plt.tight_layout()
plt.savefig(EDA_DIR / "eda_overlap_kde.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(5,3.2))
sns.histplot(data=train_eda, x="overlap", hue="label", bins=40, palette=["#ef4444","#22c55e"], alpha=0.45, ax=ax, stat="count", multiple="stack")
ax.set_title("Histogram Overlap hard negative di atas 0.7 vs hard positive di bawah 0.5")
ax.axvline(0.5, color="black", ls="--", lw=1)
ax.text(0.51, ax.get_ylim()[1]*0.9, "0.5", fontsize=8)
plt.tight_layout()
plt.savefig(EDA_DIR / "eda_overlap_hist.png", dpi=150)
plt.show()


Insight yang didapat
Mean overlap label Sesuai sekitar 0.796 dan median 0.80 yang berarti judul hampir selalu muncul di isi. Mean overlap label Tidak Sesuai sekitar 0.749 pada dataset terbaru dengan median 0.75. Angka ini jauh lebih tinggi dibanding versi lama yang hanya 0.339 sehingga tingkat kesulitan naik drastis. Hard negative dengan overlap di atas 0.7 berjumlah 954 baris atau sekitar 66 persen dari kelas Tidak Sesuai. Hard positive dengan overlap di bawah 0.5 berjumlah 278 baris atau sekitar 2 persen dari kelas Sesuai.

Dampak ke pipeline
Overlap saja tidak cukup. Selisih mean yang tipis membuat classifier berbasis overlap murni gagal pada subset sulit. Pipeline menambah fitur yang lebih tajam yaitu jendela selaras ALI n gram verbatim NGR urutan Kendall tau ORD dukungan eksplisit SUP dan perbandingan berpasangan DUP NDP RET. Fitur fitur ini dirancang untuk menangkap tukar entitas negasi yang dibalik dan urutan peran yang ditukar yang tidak terlihat pada overlap sederhana.


## 6 Kata Dominan dan TF IDF Diskriminatif

Kita melihat kata apa yang sering muncul pada judul per label serta apa yang diisi. Tujuannya untuk menilai apakah model dapat menang hanya dengan menghafal kata kunci.


In [ ]:
import re, collections

STOP_EDA = {"yang","dan","di","ke","dari","untuk","dengan","pada","adalah","akan","ini","itu","sebagai","oleh","ada","juga","tidak",
         "dalam","atau","saat","telah","sudah","hingga","karena","agar","jika","bagi","lebih","antara","secara","kepada","tersebut",
         "bahwa","kami","kita","mereka","saya","anda","satu","dua","tiga","tahun","hari","orang"}

def top_words(series, n=20):
    cnt = collections.Counter()
    for t in series.str.lower():
        for w in re.findall(r"[a-z]+", t):
            if w not in STOP_EDA and len(w)>2:
                cnt[w]+=1
    return cnt.most_common(n)

for lab, name in [(1,"sesuai"),(0,"tidak_sesuai")]:
    cw = top_words(train_eda.loc[train_eda.label==lab,"title"], 25)
    fig, ax = plt.subplots(figsize=(6,4.2))
    words, freqs = zip(*cw[::-1])
    ax.barh(words, freqs, color="#22c55e" if lab==1 else "#ef4444")
    ax.set_title(f"Top 25 Kata Judul Label {lab} {name}")
    plt.tight_layout()
    plt.savefig(EDA_DIR / f"eda_topwords_title_{name}.png", dpi=150)
    plt.show()
    print(f"top judul {name}", cw[:8])

cw_all = top_words(train_eda["content"].sample(min(8000,len(train_eda)), random_state=42), 25)
fig, ax = plt.subplots(figsize=(6,4.2))
words, freqs = zip(*cw_all[::-1])
ax.barh(words, freqs, color="#3b82f6")
ax.set_title("Top 25 Kata Isi sampel 8k")
plt.tight_layout()
plt.savefig(EDA_DIR / "eda_topwords_content.png", dpi=150)
plt.show()

# TF IDF diskriminatif
vec = TfidfVectorizer(max_features=8000, ngram_range=(1,2), min_df=5, max_df=0.85, token_pattern=r"[a-z]+", lowercase=True)
X = vec.fit_transform(train_eda["title"] + " " + train_eda["content"].str.slice(0,1200))
import scipy.sparse as sp
for lab in [0,1]:
    idx = np.where(train_eda["label"].values==lab)[0]
    mean = np.asarray(X[idx].mean(axis=0)).ravel()
    top_idx = mean.argsort()[::-1][:18]
    terms = [vec.get_feature_names_out()[i] for i in top_idx]
    scores = mean[top_idx]
    fig, ax = plt.subplots(figsize=(6,4))
    ax.barh(terms[::-1], scores[::-1], color="#22c55e" if lab==1 else "#ef4444")
    ax.set_title(f"Top TF IDF title plus content Label {lab}")
    plt.tight_layout()
    plt.savefig(EDA_DIR / f"eda_tfidf_label{lab}.png", dpi=150)
    plt.show()


Insight yang didapat
Kata dominan pada judul untuk kedua label sama sama didominasi oleh corona covid positif kasus vaksin dan varian serupa. Hal ini menandakan domain Covid yang berat sekitar 30 persen judul. TF IDF rata rata per kelas juga tidak menunjukkan pemisahan yang kuat. Perbedaan skor TF IDF antar label tipis sehingga fitur leksikal saja kurang bertenaga.

Dampak ke pipeline
Model tidak boleh hanya mengandalkan TF IDF. Pipeline menambahkan sumber daya korpus berupa deteksi entitas dari kapitalisasi cap rate ko okurensi dan Word2Vec yang dilatih sendiri untuk menangkap sinonim dan kemiripan kontekstual. Fitur leksikon wilayah dan nama orang juga ditambahkan agar pertukaran entitas lebih mudah terdeteksi.


## 7 Perbandingan Train dan Test

Kita membandingkan distribusi panjang dan overlap antara train dan test untuk menilai apakah asumsi i i d wajar dan apakah ada pergeseran distribusi.


In [ ]:
fig, axes = plt.subplots(2,2, figsize=(7,5))
sns.histplot(test_eda["title_wc"], bins=30, ax=axes[0,0], color="#64748b")
axes[0,0].set_title("Test Judul jumlah kata")
sns.histplot(test_eda["content_wc"], bins=30, ax=axes[0,1], color="#64748b")
axes[0,1].set_title("Test Isi jumlah kata")
sns.histplot(test_eda["overlap"], bins=30, ax=axes[1,0], color="#64748b")
axes[1,0].set_title("Test Overlap")
axes[1,1].axis("off")
axes[1,1].text(0.05,0.8,"Train 14397\nTest 3603\nKolom id title content\nRentang id terpisah", fontsize=9, va="top")
plt.suptitle("Distribusi Test vs Train")
plt.tight_layout()
plt.savefig(EDA_DIR / "eda_test_dist.png", dpi=150)
plt.show()

print(f"mean overlap train {train_eda.overlap.mean():.3f} test {test_eda.overlap.mean():.3f}")
print(f"mean title wc train {train_eda.title_wc.mean():.1f} test {test_eda.title_wc.mean():.1f}")
print(f"mean content wc train {train_eda.content_wc.mean():.1f} test {test_eda.content_wc.mean():.1f}")


Insight yang didapat
Distribusi panjang judul isi dan overlap pada test mirip dengan train. Mean overlap test sekitar 0.787 dan train sekitar 0.788 sehingga tidak ada pergeseran besar. Rentang id terpisah dan tidak ada kebocoran label di test.

Dampak ke pipeline
Asumsi i i d wajar sehingga validasi stratified group dapat dianggap mewakili test. Namun pipeline tetap menjaga kehati hatian dengan memakai group yang memisahkan judul atau isi yang identik agar validasi tidak bocor dan threshold tidak overfit pada pola duplikasi.


## 8 Contoh Kasus Manual

Kita melihat beberapa baris nyata untuk memahami pola Sesuai Tidak Sesuai dan hard negative yang menjadi tantangan utama.


In [ ]:
def show_examples(df, mask, label_name, n=2):
    sub = df[mask].head(n)
    for _, r in sub.iterrows():
        print(f"id {r['id']} label {r['label']} overlap {r['overlap']:.2f}")
        print(f" judul {r['title'][:120]}")
        print(f" isi {r['content'][:160].replace(chr(10),' ')}...")
        print()

print("Contoh Sesuai overlap tinggi")
show_examples(train_eda, train_eda.label==1, "sesuai", 2)
print("Contoh Tidak Sesuai topik lompat")
show_examples(train_eda, train_eda.label==0, "tidak sesuai", 2)
print("Contoh Hard Negative overlap di atas 0.7")
show_examples(train_eda, hard_neg, "hard neg", 2)
print("Contoh Hard Positive overlap di bawah 0.5")
show_examples(train_eda, hard_pos, "hard pos", 2)

# tabel ringkas untuk laporan
tbl = train_eda[hard_neg].head(2)[["id","title","content","overlap"]].copy()
tbl["content"] = tbl["content"].str.slice(0,80) + "..."
display(tbl)


Insight yang didapat
Contoh Sesuai biasanya menunjukkan judul yang hampir verbatim dengan kalimat awal isi. Contoh Tidak Sesuai ada yang jelas berbeda topik tetapi ada pula hard negative yang judul dan isinya memakai banyak kata yang sama hanya entitas atau angka yang ditukar. Pada hard negative tersebut overlap tinggi tetapi makna berubah total.

Dampak ke pipeline
Pola ini menegaskan perlunya fitur yang memeriksa konsistensi entitas dan urutan. Blok CLZ yang memeriksa siapa pengisi slot yang sah menurut isi serta blok ALG yang memeriksa monotonisitas penyelarasan dirancang untuk menangani contoh hard tersebut. Blok perbandingan berpasangan DUP dan RET juga membantu karena satu isi yang dipakai dua judul kandidat dapat dibandingkan secara langsung.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Preprocessing <a name="6"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Preprocessing pada notebook ini tidak mengubah resep model. Tujuannya adalah memastikan input ke pipeline konsisten dan dapat diulang. Langkah utama adalah normalisasi teks dan pembentukan grup untuk validasi yang jujur.


## Verifikasi Output dan Grouping Wajib

Kita membangun grup dengan union find atas judul ternormalisasi dan isi ternormalisasi. Baris yang berbagi judul atau isi digabung dalam satu grup. Grup ini dipakai untuk StratifiedGroupKFold sehingga pasangan asli dan korupsinya tidak terbelah antar fold.


In [ ]:
DATA_DIR_FALLBACK = DATA_DIR
if not (DATA_DIR_FALLBACK / "train.csv").exists():
    # coba backup
    for cand in [PROJECT_ROOT / "data" / "_v1_backup", PROJECT_ROOT / "dataOld", PROJECT_ROOT / "data"]:
        if (cand / "train.csv").exists():
            DATA_DIR_FALLBACK = cand
            break

def norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())

tr = pd.read_csv(DATA_DIR_FALLBACK / "train.csv")
te = pd.read_csv(DATA_DIR_FALLBACK / "test.csv")
full = pd.concat([tr.assign(src="tr"), te.assign(src="te", label=-1)], ignore_index=True)
full["nt"] = full["title"].map(norm)
full["nc"] = full["content"].map(norm)

NTR = len(tr)
y = tr["label"].to_numpy().astype(int)
print(f"train {tr.shape} test {te.shape} total {len(full)}")
print(f"distribusi label {tr['label'].value_counts(normalize=True).round(4).to_dict()}")

def connected_groups(df, keys=("nt", "nc")):
    parent = list(range(len(df)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for key in keys:
        d = defaultdict(list)
        for i, v in enumerate(df[key].to_numpy()):
            d[v].append(i)
        for v in d.values():
            r0 = find(v[0])
            for x in v[1:]:
                rx = find(x)
                if rx != r0:
                    parent[rx] = r0
    return np.array([find(i) for i in range(len(df))])

# ATURAN 8 untuk deliverable patuh grup test diberi id unik negatif agar tidak ikut membentuk struktur korpus
GROUP_TR = connected_groups(full.iloc[:NTR].reset_index(drop=True))
GROUP = np.concatenate([GROUP_TR, -1 - np.arange(len(full) - NTR)])
print(f"grup train unik {len(set(GROUP_TR))} test diberi grup unik tidak ikut fitting")

TTOK = [t.split() for t in full["nt"]]
CTOK = [c.split() for c in full["nc"]]
RAW_T = full["title"].astype(str).to_list()
RAW_C = full["content"].astype(str).to_list()

# statistik tambahan untuk audit
vc = full["nc"].value_counts()
share = full["nc"].map(vc).gt(1).mean()
print(f"baris yang isinya dipakai baris lain {share:.2%}  dasar perbandingan berpasangan")


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Modelling <a name="7"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Bagian ini memuat konfigurasi eksperimen pemuatan leksikon pembangunan sumber daya korpus serta seluruh blok fitur. Semua bobot diinisialisasi acak dan dilatih sendiri tanpa checkpoint pra latih.


## Seed Strict

Sel ini memastikan setiap rerun memakai seed dan pengaturan deterministik yang sama.


In [ ]:
def set_seed_strict(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass
set_seed_strict(SEED)
tick("seed strict dikunci")


## Konfigurasi Eksperimen

Konfigurasi terpusat agar mudah diaudit dan diulang.


In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    seed: int = 42
    n_splits: int = 5
    n_threads: int = 8
    model_seeds: tuple = (0, 1)
    topk_ret: int = 12
    max_train_for_tfidf: int = 400000
    w2v_dim: int = 150
    w2v_window: int = 6
    w2v_epochs: int = 6

cfg = Config()
print(cfg)


## Leksikon Eksternal train only

Leksikon wilayah hierarki nama orang KBBA stopword dan tesaurus dimuat hanya dari kosakata train agar tidak ada kebocoran dari test. Singkatan wilayah dibatasi level provinsi dan kabupaten atau kota untuk menghindari sampah seperti pmi menjadi pasar minggu.


In [ ]:
DIRECTION = {"selatan","utara","timur","barat","tengah","pusat","raya","baru","lama",
             "besar","kecil","hulu","hilir","tinggi","induk","kepulauan"}
GENERIC = {"kabupaten","kota","kec","kecamatan","kab","provinsi","prov","daerah",
           "istimewa","khusus","ibukota"}

def can_abbrev(a, words):
    n = len(words)
    if n < 2 or len(a) < 3:
        return False
    def rec(pos, wi):
        if pos == len(a):
            return wi == n
        if wi >= n:
            return False
        w = words[wi]
        for L in range(min(len(w), len(a) - pos), 0, -1):
            if a[pos:pos+L] == w[:L] and rec(pos+L, wi+1):
                return True
        return False
    return rec(0, 0)

def load_lexicons(vocab):
    prov = pd.read_csv(EXT_DIR / "provinces.csv", header=None, names=["id","name"])
    reg  = pd.read_csv(EXT_DIR / "regencies.csv", header=None, names=["id","prov","name"])
    dis  = pd.read_csv(EXT_DIR / "districts.csv", header=None, names=["id","reg","name"])
    reg2prov = dict(zip(reg["id"], reg["prov"]))
    tok2prov, tok2level, phrases12 = defaultdict(set), {}, {}
    def add(name, level, pid):
        parts = [w for w in norm(re.sub(r"[^A-Za-z ]", " ", str(name))).split() if w not in GENERIC]
        if not parts:
            return
        if level <= 2:
            phrases12[" ".join(parts)] = (level, pid)
        for w in parts:
            if len(w) < 4 or w in DIRECTION:
                continue
            tok2prov[w].add(pid)
            tok2level[w] = min(tok2level.get(w, 9), level)
    for _, r in prov.iterrows(): add(r["name"], 1, int(r["id"]))
    for _, r in reg.iterrows():  add(r["name"], 2, int(r["prov"]))
    for _, r in dis.iterrows():  add(r["name"], 3, reg2prov.get(int(r["reg"]), -1))
    by_first = defaultdict(list)
    for ph in sorted(phrases12):
        lvl, pid = phrases12[ph]
        parts = ph.split()
        if len(parts) >= 2:
            by_first[parts[0][0]].append((ph, parts, lvl, pid))
    abbr = {}
    for t in sorted(vocab):
        if not (3 <= len(t) <= 10) or t in tok2level:
            continue
        for ph, parts, lvl, pid in by_first.get(t[0], ()):
            if can_abbrev(t, parts):
                abbr[t] = (ph, lvl, pid)
                break
    def io_open(p):
        with open(p, encoding="utf-8", errors="ignore") as fh:
            return fh.read().split("\n")
    names = set()
    for line in io_open(EXT_DIR / "id-names.txt"):
        for w in norm(line).split():
            if len(w) >= 4:
                names.add(w)
    kbba = {}
    for line in io_open(EXT_DIR / "kbba.txt"):
        p = line.rstrip("\n").split("\t")
        if len(p) >= 2 and p[0] and p[1]:
            kbba[norm(p[0])] = norm(p[1])
    stop = {norm(x) for x in io_open(EXT_DIR / "stopwords-id.txt") if x.strip()}
    import json as _json
    syn = {}
    with open(EXT_DIR / "tesaurus.json", encoding="utf-8") as fh:
        for k, v in _json.load(fh).items():
            clean = set()
            for x in (v.get("sinonim") or []):
                x = re.sub(r"^(ki|kl|ark)\s+", "", norm(re.sub(r"\(.*?\)", " ", x)))
                clean.update(w for w in x.split() if len(w) >= 4)
            if clean:
                syn[norm(k)] = clean
    return dict(tok2prov=dict(tok2prov), tok2level=tok2level, abbr=abbr, names=names,
                kbba=kbba, stop=stop, syn=syn, geo_tok=set(tok2level) | set(abbr))

VOCAB_ALL = Counter()
for s in TTOK[:NTR] + CTOK[:NTR]:
    VOCAB_ALL.update(s)
LEX = load_lexicons(set(VOCAB_ALL))
tick(f"leksikon gazetteer {len(LEX['tok2level'])} singkatan {len(LEX['abbr'])} nama {len(LEX['names'])} kbba {len(LEX['kbba'])} tesaurus {len(LEX['syn'])}")
print("contoh singkatan bersih", {k: LEX["abbr"][k][0] for k in ["jabar","jatim","jateng","dki","sumut","diy"] if k in LEX["abbr"]})
junk = [j for j in ["pmi","malam","sebaru","tent"] if j in LEX["abbr"]]
print("cek sampah singkatan", "LULUS tidak ada" if not junk else f"PERINGATAN masih lolos {junk}")


## Sumber Daya Korpus Entitas IDF Word2Vec train only

cap rate dihitung dari proporsi kemunculan token yang berhuruf besar di tengah teks. Karena tanda baca sudah dibuang oleh panitia tidak ada awal kalimat yang mengacaukan sehingga cap rate di atas 0.6 menjadi detektor nama diri yang murah dan tanpa data eksternal. Word2Vec dilatih dari nol dengan workers satu.


In [ ]:
STOPW = set("di ke dari dan atau yang untuk pada dengan ini itu akan sudah telah bisa dapat ada adalah oleh saat kata ujar tapi tetapi juga karena lebih para se sang si jadi soal bagi hingga sampai agar supaya kalau jika namun serta atas bawah dalam luar antara per kini masih sedang usai jelang buat punya mau ingin harus perlu boleh a an the of in on at is as by".split())
NEGW = set("tak tidak bukan belum jangan tanpa non nggak gak enggak".split())
SUFFIXES = ["kannya","annya","inya","nya","kan","an","lah","kah","pun"]
PREFIXES = ["memper","member","menge","meny","meng","mem","men","me","diper","di","ber","ter","pen","pem","per","pe","se","ke"]

def stem(w):
    if len(w) <= 4: return w
    x = w
    for s in SUFFIXES:
        if x.endswith(s) and len(x) - len(s) >= 4:
            x = x[:-len(s)]; break
    for p in PREFIXES:
        if x.startswith(p) and len(x) - len(p) >= 4:
            x = x[len(p):]; break
    return x

def char3(w):
    w = f"^{w}$"
    return {w[i:i+3] for i in range(max(len(w) - 2, 1))}

def build_resources(raw_titles, raw_contents, tag, w2v_epochs=6):
    ntoks = [norm(c).split() for c in raw_contents]
    ttoks = [norm(t).split() for t in raw_titles]
    cap, tot = Counter(), Counter()
    for c in raw_contents:
        toks = str(c).split()
        for k, w in enumerate(toks):
            if k == 0 or not w[:1].isalpha():
                continue
            lw = w.lower(); tot[lw] += 1
            if w[0].isupper(): cap[lw] += 1
    caprate = {w: cap[w] / n for w, n in tot.items() if n >= 3}
    ENT = {w for w, r in caprate.items() if r >= 0.6 and w not in STOPW and len(w) > 1}
    phr = Counter()
    for toks in ntoks:
        for i in range(len(toks) - 1):
            if toks[i] in ENT and toks[i+1] in ENT:
                phr[(toks[i], toks[i+1])] += 1
    phrs = sorted(p for p, c in phr.items() if c >= 5)
    by_first = defaultdict(list)
    for p in phrs:
        by_first[p[0][0]].append(p)
    ACRO = defaultdict(set)
    for a in sorted(w for w in ENT if 2 <= len(w) <= 10):
        for p in by_first.get(a[0], ()):
            if can_abbrev(a, list(p)):
                ACRO[a] |= set(p)
    dfreq = Counter()
    for toks in ntoks:
        dfreq.update(set(toks) & ENT)
    COMMON = {w for w, c in dfreq.items() if c >= 12}
    co = Counter()
    for toks in ntoks:
        s = sorted(set(toks) & COMMON)[:70]
        for i in range(len(s)):
            for j in range(i + 1, len(s)):
                co[(s[i], s[j])] += 1
    dfc = Counter()
    for d in ntoks: dfc.update(set(d))
    N = len(ntoks)
    idf = {w: float(np.log(N / (1 + c))) for w, c in dfc.items()}
    from gensim.models import Word2Vec
    w2v = Word2Vec([list(x) for x in ntoks] + [list(x) for x in ttoks],
                   vector_size=cfg.w2v_dim, window=cfg.w2v_window, min_count=3, sg=1, negative=8,
                   epochs=w2v_epochs, workers=1, seed=SEED)
    tick(f"resources {tag} ENT {len(ENT)} ACRO {len(ACRO)} w2v vocab {len(w2v.wv)} N {N}")
    return dict(ENT=ENT, ACRO=dict(ACRO), caprate=caprate, dfreq=dfreq, co=co, N=N,
                idf=idf, DEF=float(np.log(N)), wv=w2v.wv, scache={})

RES = build_resources(RAW_T[:NTR], RAW_C[:NTR], "lomba train only")


## Blok Fitur Semantik LEX, ALI, NGR, ORD, SUP, ENT, CLS, NEG, dan NUM

Blok ini dirancang per tipe korupsi. ALI memakai jendela selaras dan menjadi sinyal tunggal terkuat di antara blok semantik. NGR memeriksa n gram verbatim yang rusak bila urutan diacak. ORD memakai Kendall tau posisi judul versus kemunculan pertama di isi. ENT memeriksa entitas dan prominensinya. CLZ memeriksa cloze slot yaitu siapa pengisi yang sah menurut isi. NEG memeriksa ketidakcocokan negasi.


In [ ]:
def row_features(raw_title, raw_content, R):
    ENT, ACRO, idf, DEF = R["ENT"], R["ACRO"], R["idf"], R["DEF"]
    co, dfreq, N, WV, scache = R["co"], R["dfreq"], R["N"], R["wv"], R["scache"]
    def st(w):
        if w not in scache: scache[w] = stem(w)
        return scache[w]
    def sim(a, b):
        try: return float(WV.similarity(a, b))
        except KeyError: return np.nan
    def pmi(a, b):
        if a == b: return 5.0
        c = co.get((a, b)) or co.get((b, a))
        if not c or a not in dfreq or b not in dfreq: return np.nan
        return float(np.log((c / N) / ((dfreq[a] / N) * (dfreq[b] / N))))
    T = norm(raw_title).split(); C = norm(raw_content).split()
    rt = str(raw_title).split()
    cs = set(C); cstem = {st(w) for w in cs}
    allpos = defaultdict(list)
    for k, w in enumerate(C): allpos[w].append(k)
    first = {w: p[0] for w, p in allpos.items()}
    lc = max(len(C), 1); cnt = Counter(C)
    Tc = [w for w in T if w not in STOPW]
    c3index = defaultdict(set)
    for w in sorted(cs):
        for gg in char3(w): c3index[gg].add(w)
    cent = sorted(((w, cnt[w]) for w in cs if w in ENT), key=lambda x: (-x[1], x[0]))
    top_ent = [w for w, _ in cent[:8]]
    max_ec = cent[0][1] if cent else 0
    L = min(max(3 * len(T), 45), lc)
    tset = set(Tc); best_s, best_h = 0, -1
    for s in range(0, max(lc - L + 1, 1), 3):
        h = sum(1 for w in set(C[s:s+L]) if w in tset)
        if h > best_h: best_h, best_s = h, s
    win = C[best_s:best_s+L]; winset = set(win)
    win_cov = float(np.mean([w in winset for w in Tc])) if Tc else 0.0
    win_cov_ent = (float(np.mean([w in winset for w in Tc if w in ENT])) if any(w in ENT for w in Tc) else np.nan)
    pos_hit = sorted(first[w] for w in Tc if w in first)
    span = (pos_hit[-1] - pos_hit[0]) / lc if len(pos_hit) >= 2 else np.nan
    span80 = np.nan
    if len(pos_hit) >= 3:
        k80 = max(int(0.8 * len(pos_hit)), 2)
        span80 = min(pos_hit[a+k80-1] - pos_hit[a] for a in range(len(pos_hit)-k80+1)) / lc
    cbig = set(zip(C, C[1:])); ctri = set(zip(C, C[1:], C[2:]))
    tb = list(zip(T, T[1:])); tt3 = list(zip(T, T[1:], T[2:]))
    tbc = list(zip(Tc, Tc[1:]))
    big_v = float(np.mean([b in cbig for b in tb])) if tb else np.nan
    tri_v = float(np.mean([b in ctri for b in tt3])) if tt3 else np.nan
    bigc_v = float(np.mean([b in cbig for b in tbc])) if tbc else np.nan
    xs, ys = [], []
    for k, w in enumerate(T):
        if w in first and w not in STOPW: xs.append(k); ys.append(first[w])
    ord_tau = ord_inv = np.nan
    if len(xs) >= 4:
        n_ = len(xs)
        bad = sum(1 for a in range(n_) for b in range(a+1, n_) if ys[a] > ys[b])
        ord_inv = bad / (n_ * (n_ - 1) / 2); ord_tau = 1 - 2 * ord_inv
    okp = totp = 0
    for a, b in zip(Tc, Tc[1:]):
        if a in allpos and b in allpos:
            totp += 1
            if any(pb > pa and pb - pa <= 25 for pa in allpos[a][:20] for pb in allpos[b][:20]):
                okp += 1
    ord_pair = okp / totp if totp else np.nan
    te_ = [w for w in Tc if w in ENT]; oke = tote = 0
    for a, b in zip(te_, te_[1:]):
        if a in allpos and b in allpos:
            tote += 1
            if any(pb > pa and pb - pa <= 40 for pa in allpos[a][:20] for pb in allpos[b][:20]):
                oke += 1
    ord_entpair = oke / tote if tote else np.nan
    sup_ex = sup_al = sup_sm = sup_c3 = 0
    unsup, unsup_ent = [], []
    for w in Tc:
        if w in cs: sup_ex += 1; continue
        if any(x in cs for x in sorted(ACRO.get(w, ()))): sup_al += 1; continue
        if st(w) in cstem: sup_sm += 1; continue
        cand = set()
        for gg in char3(w): cand |= c3index.get(gg, set())
        aw = char3(w)
        if cand and max(len(aw & char3(x)) / max(len(aw | char3(x)), 1) for x in sorted(cand)) >= 0.62:
            sup_c3 += 1; continue
        unsup.append(w)
        if w in ENT: unsup_ent.append(w)
    nTc = max(len(Tc), 1)
    clz_found = 0; clz_sim, clz_pmi, clz_prom = [], [], []
    for k, w in enumerate(T):
        if w not in unsup_ent: continue
        Lc = T[k-1] if k > 0 else None
        Rc = T[k+1] if k+1 < len(T) else None
        fillers = Counter()
        if Lc in allpos:
            for p in allpos[Lc][:20]:
                if p + 1 < lc and C[p+1] in ENT: fillers[C[p+1]] += 1
        if Rc in allpos:
            for p in allpos[Rc][:20]:
                if p - 1 >= 0 and C[p-1] in ENT: fillers[C[p-1]] += 1
        if not fillers: continue
        f = max(sorted(fillers), key=lambda kk: fillers[kk])
        clz_found += 1
        s_ = sim(w, f)
        if not np.isnan(s_): clz_sim.append(s_)
        p_ = pmi(w, f)
        if not np.isnan(p_): clz_pmi.append(p_)
        clz_prom.append(cnt[f] / max(max_ec, 1))
    riv_sim, riv_pmi = [], []
    for u in unsup_ent:
        for s_ in top_ent[:5]:
            v = sim(u, s_)
            if not np.isnan(v): riv_sim.append(v)
            p_ = pmi(u, s_)
            if not np.isnan(p_): riv_pmi.append(p_)
    nt_neg = sum(1 for w in T if w in NEGW)
    nw_neg = sum(1 for w in win if w in NEGW)
    nc_head = sum(1 for w in C[:60] if w in NEGW)
    neg_before = 0
    for k, w in enumerate(Tc):
        if w in allpos:
            prev_t = Tc[k-1] if k > 0 else None
            if prev_t not in NEGW:
                for p in allpos[w][:10]:
                    if p > 0 and C[p-1] in NEGW: neg_before += 1; break
    low_neg = sum(1 for w in rt if w.lower() in NEGW and w[:1].islower())
    tn = [w for w in T if any(ch.isdigit() for ch in w)]
    num_miss = sum(1 for w in tn if w not in cs)
    return dict(
        n_tok_t=len(T), n_tok_c=lc,
        cov=float(np.mean([w in cs for w in T])) if T else 0.0,
        cov_core=float(np.mean([w in cs for w in Tc])) if Tc else 0.0,
        cov_idf=sum(idf.get(w, DEF) for w in T if w in cs) / max(sum(idf.get(w, DEF) for w in T), 1e-9),
        n_miss=sum(1 for w in T if w not in cs),
        maxidf_miss=max([idf.get(w, DEF) for w in T if w not in cs], default=0.0),
        sumidf_miss=sum(idf.get(w, DEF) for w in T if w not in cs),
        cov20=float(np.mean([w in set(C[:20]) for w in T])) if T else 0.0,
        cov40=float(np.mean([w in set(C[:40]) for w in T])) if T else 0.0,
        cov80=float(np.mean([w in set(C[:80]) for w in T])) if T else 0.0,
        cov150=float(np.mean([w in set(C[:150]) for w in T])) if T else 0.0,
        win_cov=win_cov, win_cov_ent=win_cov_ent, win_hit=best_h,
        win_pos=best_s / lc, win_len=L, span=span, span80=span80,
        win_cov_idf=sum(idf.get(w, DEF) for w in Tc if w in winset) / max(sum(idf.get(w, DEF) for w in Tc), 1e-9),
        big_v=big_v, tri_v=tri_v, bigc_v=bigc_v,
        ord_tau=ord_tau, ord_inv=ord_inv, ord_pair=ord_pair,
        ord_pair_n=totp, ord_entpair=ord_entpair, ord_entpair_n=tote,
        sup_ex=sup_ex / nTc, sup_al=sup_al, sup_sm=sup_sm, sup_c3=sup_c3,
        n_unsup=len(unsup),
        unsup_maxidf=max([idf.get(w, DEF) for w in unsup], default=0.0),
        ent_n_t=sum(1 for w in Tc if w in ENT),
        ent_frac_t=float(np.mean([w in ENT for w in Tc])) if Tc else 0.0,
        ent_n_unsup=len(unsup_ent),
        ent_unsup_frac=len(unsup_ent) / max(sum(1 for w in Tc if w in ENT), 1),
        ent_unsup_maxidf=max([idf.get(w, DEF) for w in unsup_ent], default=0.0),
        ent_n_c=len(cent),
        pro_mean=float(np.mean([cnt[w] / max(max_ec, 1) for w in Tc if w in cs] or [0])),
        pro_min=float(np.min([cnt[w] / max(max_ec, 1) for w in Tc if w in cs] or [0])),
        fp_mean=float(np.mean([first[w] / lc for w in Tc if w in first] or [1.0])),
        fp_max=float(np.max([first[w] / lc for w in Tc if w in first] or [1.0])),
        first_ent_match=float(1.0 if (top_ent and top_ent[0] in T) else 0.0),
        ent_head20=float(np.mean([w in set(C[:20]) for w in Tc if w in ENT]) if any(w in ENT for w in Tc) else np.nan),
        clz_found=clz_found,
        clz_sim_max=max(clz_sim) if clz_sim else np.nan,
        clz_sim_mean=float(np.mean(clz_sim)) if clz_sim else np.nan,
        clz_pmi_max=max(clz_pmi) if clz_pmi else np.nan,
        clz_prom_max=max(clz_prom) if clz_prom else np.nan,
        riv_sim_max=max(riv_sim) if riv_sim else np.nan,
        riv_sim_mean=float(np.mean(riv_sim)) if riv_sim else np.nan,
        riv_pmi_max=max(riv_pmi) if riv_pmi else np.nan,
        riv_pmi_mean=float(np.mean(riv_pmi)) if riv_pmi else np.nan,
        riv_gap=(max(riv_sim) - max(riv_pmi) / 5.0) if (riv_sim and riv_pmi) else np.nan,
        neg_t=nt_neg, neg_win=nw_neg, neg_c_head=nc_head,
        neg_diff=nt_neg - min(nw_neg, 2),
        neg_t_missing=int(nt_neg == 0 and nw_neg > 0),
        neg_t_extra=int(nt_neg > 0 and nw_neg == 0),
        neg_before=neg_before, neg_before_frac=neg_before / nTc, low_neg=low_neg,
        n_num_t=len(tn), num_miss=num_miss, num_miss_frac=num_miss / max(len(tn), 1),
    )

rows = []
for i in range(len(full)):
    rows.append(row_features(RAW_T[i], RAW_C[i], RES))
    if i % 4000 == 0: tick(f"fitur semantik {i} dari {len(full)}")
F_SEM = pd.DataFrame(rows)
SEM_COLS = list(F_SEM.columns)
tick(f"blok semantik siap {F_SEM.shape}")


## Blok ALG Penyelarasan Monoton

Judul berita adalah ringkasan lead. Token judul biasanya muncul di isi dengan urutan yang sama. Token yang gagal masuk penyelarasan menjadi kandidat yang dikorupsi. Pertukaran peran otomatis merusak monotonisitas.


In [ ]:
import bisect

def lis_align(pairs):
    cands = [(ti, p) for ti, ps in pairs for p in ps]
    if not cands: return set(), []
    cands.sort(key=lambda x: (x[0], x[1]))
    n = len(cands); best = [1]*n; prev = [-1]*n
    for i in range(n):
        for j in range(i):
            if cands[j][0] < cands[i][0] and cands[j][1] < cands[i][1] and best[j]+1 > best[i]:
                best[i] = best[j]+1; prev[i] = j
    e = int(np.argmax(best)); chain = []
    while e != -1:
        chain.append(cands[e]); e = prev[e]
    chain.reverse()
    return {t for t, _ in chain}, chain

ENTS = RES["ENT"]; IDF = RES["idf"]; DEFI = RES["DEF"]; WV = RES["wv"]
def _sim(a, b):
    try: return float(WV.similarity(a, b))
    except KeyError: return np.nan

alg_rows = []
for i in range(len(full)):
    T, C = TTOK[i], CTOK[i]
    lc = max(len(C), 1)
    allpos = defaultdict(list)
    for k, w in enumerate(C): allpos[w].append(k)
    cnt = Counter(C)
    cent = sorted(((w, cnt[w]) for w in set(C) if w in ENTS), key=lambda x: (-x[1], x[0]))
    max_ec = cent[0][1] if cent else 1
    Tc = [(k, w) for k, w in enumerate(T) if w not in STOPW]
    pairs = [(k, allpos[w][:12]) for k, w in Tc if w in allpos]
    aligned, chain = lis_align(pairs)
    nTc = max(len(Tc), 1)
    present = {k for k, w in Tc if w in allpos}
    broken = present - aligned
    ck = [t for t, _ in chain]; cp = [p for _, p in chain]
    def expect(k):
        if not ck: return None
        j = bisect.bisect_left(ck, k)
        lo = cp[j-1] if j > 0 else None
        hi = cp[j] if j < len(cp) else None
        if lo is None: return hi
        if hi is None: return lo
        return (lo + hi) // 2
    gap_ent = gap_neg = gap_checked = 0
    gap_sim, gap_prom = [], []
    for k, w in Tc:
        if k in aligned: continue
        e = expect(k)
        if e is None: continue
        gap_checked += 1
        seg = C[max(e-6, 0):min(e+7, lc)]
        es = [x for x in seg if x in ENTS]
        if es:
            gap_ent += 1
            best_e = max(sorted(set(es)), key=lambda x: cnt[x])
            s_ = _sim(w, best_e)
            if not np.isnan(s_): gap_sim.append(s_)
            gap_prom.append(cnt[best_e] / max_ec)
        if any(x in NEGW for x in seg) and w not in NEGW: gap_neg += 1
    span = (cp[-1] - cp[0]) / lc if len(cp) >= 2 else np.nan
    density = len(chain) / max(cp[-1]-cp[0]+1, 1) if len(cp) >= 2 else np.nan
    alg_rows.append(dict(
        algn_n=len(aligned), algn_frac=len(aligned)/nTc,
        algn_frac_ent=(len([k for k, w in Tc if k in aligned and w in ENTS]) / max(len([1 for _, w in Tc if w in ENTS]), 1)),
        algn_broken=len(broken), algn_broken_frac=len(broken)/nTc,
        algn_unaligned=nTc-len(aligned), algn_span=span, algn_density=density,
        algn_start=cp[0]/lc if cp else np.nan,
        algn_idf_miss=sum(IDF.get(w, DEFI) for k, w in Tc if k not in aligned),
        algn_maxidf_miss=max([IDF.get(w, DEFI) for k, w in Tc if k not in aligned], default=0.0),
        gap_checked=gap_checked, gap_ent=gap_ent,
        gap_ent_frac=gap_ent/max(gap_checked, 1),
        gap_sim_max=max(gap_sim) if gap_sim else np.nan,
        gap_sim_mean=float(np.mean(gap_sim)) if gap_sim else np.nan,
        gap_prom_max=max(gap_prom) if gap_prom else np.nan,
        gap_neg=gap_neg, gap_neg_frac=gap_neg/max(gap_checked, 1),
    ))
    if i % 5000 == 0: tick(f"ALG {i} dari {len(full)}")
F_ALG = pd.DataFrame(alg_rows)
ALG_COLS = list(F_ALG.columns)
tick(f"blok ALG siap {F_ALG.shape}")


## Blok RET Retrieval train only

Blok ini menjawab pertanyaan apakah ada judul lain di korpus yang lebih berhak atas isi ini. Indeks TF IDF di fit hanya pada train dan seluruh grup diri dibuang dari kandidat agar tidak ada tie sempurna yang membocorkan struktur duplikasi.


In [ ]:
TOPK = 12
def rowwise_dot(A, B):
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()

def block_scores(Q, D, groups, own_vec, blocksize=800):
    n = Q.shape[0]
    nd = D.shape[0]
    out = np.zeros((n, 6 + TOPK), dtype=np.float32)
    for s in range(0, n, blocksize):
        e = min(s + blocksize, n)
        S = (Q[s:e] @ D.T).toarray()
        for r in range(e - s):
            i = s + r
            row = S[r]
            own = float(own_vec[i])
            row_m = row.copy()
            if i < nd:
                row_m[groups[:nd] == groups[i]] = -1.0
            k = min(TOPK, nd)
            top = np.argpartition(-row_m, k-1)[:k]
            top = top[np.argsort(-row_m[top])]
            tv = row_m[top]
            best = float(tv[0]) if len(tv) else 0.0
            out[i, 0] = own; out[i, 1] = best; out[i, 2] = own - best
            out[i, 3] = own / (best + 1e-9); out[i, 4] = int((row_m > own).sum())
            out[i, 5] = float(np.mean(tv[:5])) if len(tv) else 0.0
            out[i, 6:6+len(tv)] = tv
    return out

lead  = full["nc"].map(lambda x: " ".join(x.split()[:90]))
lead2 = full["nc"].map(lambda x: " ".join(x.split()[:220]))
RETF = {}
names = ["own","best","marg","ratio","n_above","top5mean"] + [f"t{i}" for i in range(TOPK)]
FIT_TXT = pd.concat([full["nt"].iloc[:NTR], lead2.iloc[:NTR]])
for tag, kw in [("w12", dict(ngram_range=(1,2), min_df=2, sublinear_tf=True, max_features=400000)),
                ("c35", dict(analyzer="char_wb", ngram_range=(3,5), min_df=3, sublinear_tf=True, max_features=400000))]:
    v = TfidfVectorizer(**kw); v.fit(FIT_TXT)
    Tm = normalize(v.transform(full["nt"].iloc[:NTR]))
    TmA = normalize(v.transform(full["nt"]))
    for qtag, q in [("lead", lead), ("full", lead2)]:
        Qm = normalize(v.transform(q))
        A = block_scores(Qm, Tm, GROUP, rowwise_dot(Qm, TmA))
        for j, nm in enumerate(names): RETF[f"ret_{tag}_{qtag}_{nm}"] = A[:, j]
        tick(f"RET {tag} {qtag}")
v = TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True, max_features=400000)
v.fit(FIT_TXT)
Qm = normalize(v.transform(full["nt"]))
Dm = normalize(v.transform(lead2.iloc[:NTR]))
CmA = normalize(v.transform(lead2))
A = block_scores(Qm, Dm, GROUP, rowwise_dot(Qm, CmA))
for j, nm in enumerate(names): RETF[f"ret_rev_{nm}"] = A[:, j]
F_RET = pd.DataFrame(RETF)
RET_COLS = list(F_RET.columns)
tick(f"blok RET siap {F_RET.shape}")


## Blok DUP dan NDP Perbandingan Berpasangan train only

Blok ini adalah terobosan skor. Alih alih bertanya apakah judul ini cocok dengan isinya ubah pertanyaannya menjadi bila isi ini punya dua judul kandidat di korpus mana yang lebih berhak. Efeknya pada single LightGBM adalah lonjakan besar tanpa memakai label tambahan.


In [ ]:
# DUP struktur duplikasi persis
import collections

# bangun indeks kembaran untuk NDP dan DUP referensi
bync = defaultdict(list)
bynt = defaultdict(list)
ncs = full["nc"].to_numpy()
nts = full["nt"].to_numpy()
for i in range(NTR):
    bync[ncs[i]].append(i)
    bynt[nts[i]].append(i)

dup_rows = []
for i in range(len(full)):
    same_title = [j for j in bynt[nts[i]] if j != i]
    same_cont  = [j for j in bync[ncs[i]] if j != i]
    t_diffc = [j for j in same_title if ncs[j] != ncs[i]]
    c_difft = [j for j in same_cont if nts[j] != nts[i]]
    dup_rows.append(dict(
        dup_title_n=len(same_title), dup_title_diffc=len(t_diffc),
        dup_content_n=len(same_cont), dup_content_difft=len(c_difft),
        dup_has_title_rival=int(len(t_diffc) > 0),
        dup_has_content_rival=int(len(c_difft) > 0)))
F_DUP = pd.DataFrame(dup_rows)
tick(f"blok DUP siap {F_DUP.shape}")

# NDP kembaran mirip plus perbandingan silang
C_HI, T_DUP, TOPN = 0.85, 0.93, 10
vC = TfidfVectorizer(ngram_range=(1,1), min_df=2, sublinear_tf=True, max_features=400000)
vC.fit(full["nc"].iloc[:NTR])
Cm_idx = normalize(vC.transform(full["nc"].iloc[:NTR]))
Cm = normalize(vC.transform(full["nc"]))
vT = TfidfVectorizer(ngram_range=(1,1), min_df=1, sublinear_tf=True, max_features=400000)
vT.fit(full["nt"].iloc[:NTR])
Tm_idx = normalize(vT.transform(full["nt"].iloc[:NTR]))
Tm2 = normalize(vT.transform(full["nt"]))
n = len(full)

def topn_index(Q, D):
    store = {}
    nd = D.shape[0]
    for s in range(0, Q.shape[0], 600):
        S = (Q[s:s+600] @ D.T).toarray()
        for r in range(S.shape[0]):
            i = s + r
            if i < nd: S[r, i] = -1
            k = min(TOPN, nd-1)
            top = np.argpartition(-S[r], k-1)[:k]
            top = top[np.argsort(-S[r][top])]
            store[i] = [(int(j), float(S[r][j])) for j in top]
    return store

nnC = topn_index(Cm, Cm_idx); tick("NDP tetangga isi siap indeks train")
nnT = topn_index(Tm2, Tm_idx); tick("NDP tetangga judul siap indeks train")
tsim = lambda i, j: float(Tm2[i].multiply(Tm_idx[j]).sum())
csim = lambda i, j: float(Cm[i].multiply(Cm_idx[j]).sum())

def cross_feats(Ttoks, Ctoks, allpos, cnt):
    cs = set(Ctoks); Tc = [w for w in Ttoks if w not in STOPW]
    if not Tc: return dict(x_cov=0.0, x_win=0.0, x_ord=np.nan, x_big=0.0, x_neg=0)
    lc = max(len(Ctoks), 1)
    L = min(max(3*len(Ttoks), 45), lc); tset = set(Tc)
    best_s, best_h = 0, -1
    for s in range(0, max(lc-L+1, 1), 6):
        h = sum(1 for w in set(Ctoks[s:s+L]) if w in tset)
        if h > best_h: best_h, best_s = h, s
    winset = set(Ctoks[best_s:best_s+L])
    first = {w: allpos[w][0] for w in tset if w in allpos}
    xs = [first[w] for w in Tc if w in first]
    ordv = np.nan
    if len(xs) >= 4:
        inv = sum(1 for a in range(len(xs)) for b in range(a+1, len(xs)) if xs[a] > xs[b])
        ordv = 1 - 2*inv/(len(xs)*(len(xs)-1)/2)
    cbig = set(zip(Ctoks, Ctoks[1:])); tb = list(zip(Ttoks, Ttoks[1:]))
    return dict(x_cov=float(np.mean([w in cs for w in Tc])),
                x_win=float(np.mean([w in winset for w in Tc])),
                x_ord=ordv,
                x_big=float(np.mean([b in cbig for b in tb])) if tb else 0.0,
                x_neg=sum(1 for w in Ttoks if w in NEGW))

NF = defaultdict(list)
for i in range(n):
    allpos = defaultdict(list)
    for k, w in enumerate(CTOK[i]): allpos[w].append(k)
    cnt = Counter(CTOK[i])
    cs_ = nnC[i]
    hi = [(j, s) for j, s in cs_ if s >= C_HI]
    rival, dupl = [], []
    for j, s in hi:
        ts = tsim(i, j)
        (dupl if ts >= T_DUP else rival).append((j, s, ts))
    NF["ndp_c_sim1"].append(cs_[0][1] if cs_ else 0.0)
    NF["ndp_c_sim2"].append(cs_[1][1] if len(cs_) > 1 else 0.0)
    NF["ndp_c_hi_n"].append(len(hi)); NF["ndp_c_dup_n"].append(len(dupl))
    NF["ndp_c_rival_n"].append(len(rival))
    NF["ndp_c_rival_tsim"].append(max([x[2] for x in rival], default=0.0))
    NF["ndp_has_rival"].append(int(len(rival) > 0))
    tsn = nnT[i]
    thi = [(j, s) for j, s in tsn if s >= T_DUP]
    tclaim = [(j, s) for j, s in thi if csim(i, j) < C_HI]
    NF["ndp_t_sim1"].append(tsn[0][1] if tsn else 0.0)
    NF["ndp_t_hi_n"].append(len(thi)); NF["ndp_t_claim_n"].append(len(tclaim))
    NF["ndp_t_claimed"].append(int(len(tclaim) > 0))
    own = cross_feats(TTOK[i], CTOK[i], allpos, cnt)
    rsorted = [(j, s, tsim(i, j)) for j, s in cs_[:2]]
    NF["xr_sim1"].append(rsorted[0][1] if rsorted else 0.0)
    NF["xr_tsim1"].append(rsorted[0][2] if rsorted else 0.0)
    for slot in range(2):
        pre = "xr" if slot == 0 else "xr2"
        if slot < len(rsorted):
            riv = cross_feats(TTOK[rsorted[slot][0]], CTOK[i], allpos, cnt)
            for k_ in own:
                if slot == 0: NF[f"xr_{k_}_own"].append(own[k_])
                NF[f"{pre}_{k_}_riv"].append(riv[k_])
                NF[f"{pre}_{k_}_d"].append(own[k_] - riv[k_])
            NF[f"{pre}_win_better"].append(int(riv["x_win"] > own["x_win"]))
            NF[f"{pre}_cov_better"].append(int(riv["x_cov"] > own["x_cov"]))
        else:
            for k_ in own:
                if slot == 0: NF[f"xr_{k_}_own"].append(own[k_])
                NF[f"{pre}_{k_}_riv"].append(np.nan); NF[f"{pre}_{k_}_d"].append(np.nan)
            NF[f"{pre}_win_better"].append(-1); NF[f"{pre}_cov_better"].append(-1)
    rd = [(j, s, tsim(i, j)) for j, s in cs_ if tsim(i, j) < T_DUP]
    NF["xrd_sim"].append(rd[0][1] if rd else 0.0)
    NF["xrd_tsim"].append(rd[0][2] if rd else 0.0)
    if rd:
        riv = cross_feats(TTOK[rd[0][0]], CTOK[i], allpos, cnt)
        for k_ in own:
            NF[f"xrd_{k_}_riv"].append(riv[k_]); NF[f"xrd_{k_}_d"].append(own[k_]-riv[k_])
        NF["xrd_win_better"].append(int(riv["x_win"] > own["x_win"]))
        NF["xrd_cov_better"].append(int(riv["x_cov"] > own["x_cov"]))
    else:
        for k_ in own:
            NF[f"xrd_{k_}_riv"].append(np.nan); NF[f"xrd_{k_}_d"].append(np.nan)
        NF["xrd_win_better"].append(-1); NF["xrd_cov_better"].append(-1)

F_NDP = pd.DataFrame(NF)
# simpan kolom DUP untuk dipakai pada analisis NDP dan audit
DUP_COLS = list(F_DUP.columns)
NDP_COLS = list(F_NDP.columns)
tick(f"blok NDP siap {F_NDP.shape}")


## Blok EXT Leksikon dalam Perbandingan

Blok ini menilai apakah kata wilayah nama orang atau istilah leksikon lain di judul mendapat dukungan di isi dan bagaimana saingan judul lain berperilaku pada isi yang sama.


In [ ]:
def title_vs_content(T, cs_exp, cprov_top, R):
    geo_tok, tok2level, abbr, tok2prov = R["geo_tok"], R["tok2level"], R["abbr"], R["tok2prov"]
    names, kbba, stop, syn = R["names"], R["kbba"], R["stop"], R["syn"]
    Tn = [kbba.get(w, w) for w in T]
    Tc = [w for w in Tn if w not in stop]
    tgeo = [w for w in T if w in geo_tok]
    tgeo_miss = [w for w in tgeo if w not in cs_exp and not (w in abbr and set(abbr[w][0].split()) & cs_exp)]
    tname = [w for w in T if w in names]
    tname_miss = [w for w in tname if w not in cs_exp]
    miss = [w for w in Tc if w not in cs_exp]
    syn_hit = sum(1 for w in miss if syn.get(w, ()) and set(syn[w]) & cs_exp)
    tprov = set()
    for w in tgeo:
        tprov |= tok2prov.get(w, set()) or ({abbr[w][2]} if w in abbr else set())
    prov_mismatch = int(bool(tprov) and bool(cprov_top) and not (tprov & cprov_top))
    return dict(
        e_stop_ov=float(np.mean([w in cs_exp for w in Tc])) if Tc else 0.0,
        e_miss=len(miss), e_miss_syn=len(miss)-syn_hit, e_syn_hit=syn_hit,
        e_geo_n=len(tgeo), e_geo_miss=len(tgeo_miss),
        e_geo_miss_frac=len(tgeo_miss)/max(len(tgeo), 1),
        e_geo_lvl=min([tok2level.get(w, abbr[w][1] if w in abbr else 9) for w in tgeo_miss], default=9),
        e_prov_mismatch=prov_mismatch, e_name_miss=len(tname_miss))

rival_of = np.full(n, -1, dtype=int)
for i in range(n):
    for j, s in nnC[i]:
        if tsim(i, j) < T_DUP: rival_of[i] = j; break

kbba, geo_tok, tok2prov, abbr = LEX["kbba"], LEX["geo_tok"], LEX["tok2prov"], LEX["abbr"]
ext_rows = []
for i in range(n):
    C = CTOK[i]; cs = set(C); cs_exp = set(cs)
    for w in sorted(cs):
        if w in kbba: cs_exp.update(kbba[w].split())
    cprov = Counter()
    for w in C:
        if w in geo_tok:
            for p in (tok2prov.get(w) or ({abbr[w][2]} if w in abbr else set())):
                cprov[p] += 1
    cprov_top = {p for p, _ in sorted(cprov.items(), key=lambda kv: (-kv[1], kv[0]))[:2]}
    own = title_vs_content(TTOK[i], cs_exp, cprov_top, LEX)
    rec = {f"ext_{k}_own": v for k, v in own.items()}
    j = rival_of[i]
    if j >= 0:
        riv = title_vs_content(TTOK[j], cs_exp, cprov_top, LEX)
        for k, v in own.items(): rec[f"ext_{k}_d"] = v - riv[k]
        rec["ext_geo_better_riv"] = int(riv["e_geo_miss"] < own["e_geo_miss"])
        rec["ext_stop_better_riv"] = int(riv["e_stop_ov"] > own["e_stop_ov"])
        rec["ext_prov_better_riv"] = int(riv["e_prov_mismatch"] < own["e_prov_mismatch"])
    else:
        for k in own: rec[f"ext_{k}_d"] = np.nan
        rec["ext_geo_better_riv"] = -1
        rec["ext_stop_better_riv"] = -1
        rec["ext_prov_better_riv"] = -1
    rec["ext_n_cprov"] = len(cprov)
    ext_rows.append(rec)
    if i % 5000 == 0: tick(f"EXT {i} dari {n}")
F_EXT = pd.DataFrame(ext_rows)
EXT_COLS = list(F_EXT.columns)
tick(f"blok EXT siap {F_EXT.shape}")


## Generator Negatif Sintetis untuk Cross Encoder

Resep diturunkan dari tipologi korupsi nyata yaitu tukar entitas negasi yang dibalik tukar peran dan pasangan judul dari artikel lain. Generator ini dipakai hanya untuk melatih cross encoder bukan sebagai baris latih GBDT.


In [ ]:
MODAL = set("bisa dapat akan boleh harus perlu mampu sanggup wajib efektif aman ampuh mudah siap layak berhasil sempat pernah lagi".split())
RECIPE = dict(SUB_ENT=0.574, NEG_DEL=0.106, NEG_INS=0.015, PERM=0.061, SUB_NUM=0.015, SUB_MULTI=0.061, PAIR=0.167)

def collapse(s):
    return re.sub(r"\s+", " ", str(s)).strip()

class Corruptor:
    OPS = ["SUB_ENT","NEG_DEL","NEG_INS","PERM","SUB_NUM","SUB_MULTI","PAIR"]
    def __init__(self, res, titles_raw, contents_raw, groups, seed=0):
        self.res = res; self.rng = np.random.default_rng(seed)
        self.ENT = res["ENT"]; self.wv = res["wv"]; self.neigh = {}
        self.titles_raw = list(titles_raw); self.groups = groups
        self.p_absent = 0.45; self.tf_pow = 1.0; self.pair_top = 2
        tf = Counter()
        for t in titles_raw:
            tf.update(w for w in norm(t).split() if w in self.ENT)
        self.tf = tf
        self.pool = {w for w, c in tf.items() if c >= 4}
        v = TfidfVectorizer(ngram_range=(1,1), min_df=3, sublinear_tf=True, max_features=200000)
        M = normalize(v.fit_transform([" ".join(norm(c).split()[:120]) for c in contents_raw]))
        Tt = normalize(v.transform([norm(t) for t in titles_raw]))
        self.nn = {}
        for s in range(0, M.shape[0], 1000):
            S = (M[s:s+1000] @ Tt.T).toarray()
            for r in range(S.shape[0]):
                i = s + r; S[r][groups == groups[i]] = -1
                top = np.argpartition(-S[r], 8)[:8]
                self.nn[i] = [int(j) for j in top[np.argsort(-S[r][top])]]
    def neighbors(self, w, k=60):
        if w in self.neigh: return self.neigh[w]
        out = []
        try:
            for x, _ in self.wv.most_similar(w, topn=300):
                if x in self.pool and x != w and len(x) > 1: out.append(x)
                if len(out) >= k: break
        except KeyError: pass
        self.neigh[w] = out; return out
    def sub_ent(self, T, cset, i_row):
        cand = [i for i, w in enumerate(T) if w.lower() in self.ENT and w.lower() not in STOPW]
        self.rng.shuffle(cand)
        for i in cand:
            nb = self.neighbors(T[i].lower())
            if not nb: continue
            want_absent = self.rng.random() < self.p_absent
            sel = [x for x in nb if (x not in cset) == want_absent] or nb
            idf = self.res["idf"]; i0 = idf.get(T[i].lower(), self.res["DEF"])
            wgt = np.array([self.tf.get(x, 1) for x in sel], dtype=float) ** self.tf_pow
            wgt = wgt / (1.0 + np.abs(np.array([idf.get(x, self.res["DEF"]) for x in sel]) - i0))
            new = str(sel[int(self.rng.choice(len(sel), p=wgt/wgt.sum()))])
            T2 = list(T); T2[i] = new.capitalize() if T[i][:1].isupper() else new
            return T2, "SUB_ENT"
        return None
    def sub_multi(self, T, cset, i_row):
        cand = [i for i in range(len(T)-1) if T[i].lower() in self.ENT and T[i+1].lower() in self.ENT]
        self.rng.shuffle(cand)
        for i in cand:
            nb = self.neighbors(T[i].lower()) or self.neighbors(T[i+1].lower())
            if not nb: continue
            new = str(nb[int(self.rng.integers(len(nb)))])
            return T[:i] + [new.capitalize()] + T[i+2:], "SUB_MULTI"
        return None
    def neg_del(self, T, cset, i_row):
        idx = [i for i, w in enumerate(T) if w.lower() in NEGW]
        if not idx: return None
        i = int(idx[int(self.rng.integers(len(idx)))])
        return T[:i] + T[i+1:], "NEG_DEL"
    def neg_ins(self, T, cset, i_row):
        pos = [i for i, w in enumerate(T) if i > 0 and w.lower() in MODAL and T[i-1].lower() not in NEGW]
        if not pos: return None
        i = int(pos[int(self.rng.integers(len(pos)))])
        w = ["tak","tidak","belum"][int(self.rng.integers(3))]
        return T[:i] + [w] + T[i:], "NEG_INS"
    def perm(self, T, cset, i_row):
        cand = [i for i, w in enumerate(T) if w.lower() in self.ENT and w.lower() not in STOPW]
        if len(cand) < 2: return None
        a, b = self.rng.choice(cand, size=2, replace=False)
        T2 = list(T); T2[a], T2[b] = T2[b], T2[a]
        return T2, "PERM"
    def sub_num(self, T, cset, i_row):
        cand = [i for i, w in enumerate(T) if w.isdigit() and len(w) >= 2]
        if not cand: return None
        i = int(cand[int(self.rng.integers(len(cand)))]); v = int(T[i])
        d = max(1, int(abs(v) * self.rng.uniform(0.05, 0.6)))
        new = v + d * (1 if self.rng.random() < 0.5 else -1)
        T2 = list(T); T2[i] = str(new if new > 0 else v + d)
        return T2, "SUB_NUM"
    def pair(self, T, cset, i_row):
        nb = self.nn.get(i_row)
        if not nb: return None
        j = int(nb[int(self.rng.integers(min(self.pair_top, len(nb))))])
        return collapse(self.titles_raw[j]).split(), "PAIR"
    def corrupt(self, raw_title, raw_content, i_row, recipe=RECIPE):
        T = collapse(raw_title).split(); cset = set(norm(raw_content).split())
        fns = dict(SUB_ENT=self.sub_ent, NEG_DEL=self.neg_del, NEG_INS=self.neg_ins, PERM=self.perm, SUB_NUM=self.sub_num, SUB_MULTI=self.sub_multi, PAIR=self.pair)
        p = np.array([recipe[o] for o in self.OPS], dtype=float); p /= p.sum()
        order = list(self.rng.choice(self.OPS, size=len(self.OPS), replace=False, p=p))
        for o in order:
            r = fns[o](T, cset, i_row)
            if r is not None and r[0] != T: return " ".join(r[0]), r[1]
        return None, None

CORR = Corruptor(RES, RAW_T[:NTR], RAW_C[:NTR], GROUP[:NTR], seed=7)
tick("corruptor siap")


## Definisi Cross Encoder from scratch

Cross encoder memakai BiGRU dengan soft alignment ala ESIM. Embedding diinisialisasi acak 200 dimensi dan dilatih sendiri. Model ini peka terhadap urutan sehingga mampu menangkap korupsi permutasi dan negasi.


In [ ]:
import torch.nn as nn_mod
import torch.nn.functional as Fn

MAXT, MAXC = 24, 200

def prep_txt(s):
    s = re.sub(r"(?<=\d)[.,](?=\d{3}\b)", " ", str(s))
    s = re.sub(r"[^A-Za-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

class Enc(nn_mod.Module):
    def __init__(self, V, d=200, h=160):
        super().__init__()
        self.emb = nn_mod.Embedding(V, d, padding_idx=0)
        nn_mod.init.normal_(self.emb.weight, 0, 0.1)
        with torch.no_grad(): self.emb.weight[0].zero_()
        self.gru_t = nn_mod.GRU(d+1, h, batch_first=True, bidirectional=True)
        self.gru_c = nn_mod.GRU(d, h, batch_first=True, bidirectional=True)
        self.proj = nn_mod.Sequential(nn_mod.Linear(8*h, 2*h), nn_mod.ReLU(), nn_mod.Dropout(0.2))
        self.gru_v = nn_mod.GRU(2*h, h, batch_first=True, bidirectional=True)
        self.head = nn_mod.Sequential(nn_mod.Linear(6*h, 160), nn_mod.ReLU(), nn_mod.Dropout(0.2), nn_mod.Linear(160, 1))
    def forward(self, t, c, tflag):
        mt = (t > 0).float(); mc = (c > 0).float()
        ht, _ = self.gru_t(torch.cat([self.emb(t), tflag.unsqueeze(-1)], -1))
        hc, _ = self.gru_c(self.emb(c))
        att = torch.bmm(ht, hc.transpose(1, 2)).masked_fill(mc.unsqueeze(1) == 0, -1e4)
        a = torch.bmm(Fn.softmax(att, -1), hc)
        v, _ = self.gru_v(self.proj(torch.cat([ht, a, ht-a, ht*a], -1)))
        vm = v * mt.unsqueeze(-1)
        avg = vm.sum(1) / mt.sum(1, keepdim=True).clamp(min=1)
        mx = vm.masked_fill(mt.unsqueeze(-1) == 0, -1e4).max(1).values
        cm = hc * mc.unsqueeze(-1)
        cavg = cm.sum(1) / mc.sum(1, keepdim=True).clamp(min=1)
        return self.head(torch.cat([avg, mx, cavg], -1)).squeeze(-1)

def encode(titles, contents, vocab):
    m = len(titles)
    T = np.zeros((m, MAXT), dtype=np.int64); C = np.zeros((m, MAXC), dtype=np.int64)
    Fl = np.zeros((m, MAXT), dtype=np.float32)
    for i, (t, c) in enumerate(zip(titles, contents)):
        tt = norm(t).split()[:MAXT]; cc = norm(c).split()[:MAXC]; cset = set(cc)
        for j, w in enumerate(tt):
            T[i, j] = vocab.get(w, 1); Fl[i, j] = 1.0 if w in cset else 0.0
        for j, w in enumerate(cc): C[i, j] = vocab.get(w, 1)
    return T, C, Fl

@torch.no_grad()
def nn_predict(m, T, C, Fl, bs=256):
    m.eval(); out = []
    for s in range(0, len(T), bs):
        out.append(torch.sigmoid(m(torch.tensor(T[s:s+bs]).to(DEV), torch.tensor(C[s:s+bs]).to(DEV), torch.tensor(Fl[s:s+bs]).to(DEV))).cpu().numpy())
    return np.concatenate(out)

def nn_train(m, Xtr, ytr, Xva, yva, epochs, lr, seed, bs=128):
    g = torch.Generator(device="cpu"); g.manual_seed(seed)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    T, C, Fl = Xtr; nn_ = len(ytr)
    steps = max(nn_ // bs, 1) * epochs
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    yt = torch.tensor(ytr, dtype=torch.float32)
    best, best_state, step = -1.0, None, 0
    rng = np.random.default_rng(seed)
    for ep in range(epochs):
        m.train(); idx = rng.permutation(nn_)
        for s in range(0, nn_ - bs + 1, bs):
            b = idx[s:s+bs]
            loss = Fn.binary_cross_entropy_with_logits(m(torch.tensor(T[b]).to(DEV), torch.tensor(C[b]).to(DEV), torch.tensor(Fl[b]).to(DEV)), yt[b].to(DEV))
            opt.zero_grad(); loss.backward()
            nn_mod.utils.clip_grad_norm_(m.parameters(), 2.0); opt.step()
            if step < steps - 1: sch.step()
            step += 1
        a = roc_auc_score(yva, nn_predict(m, *Xva)) if len(set(yva)) > 1 else float("nan")
        if a > best:
            best = a
            best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
    if best_state: m.load_state_dict(best_state)
    return best

COMP_T = [prep_txt(x) for x in RAW_T]
COMP_C = [prep_txt(x) for x in RAW_C]
FOLDS = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42).split(np.zeros(NTR), y, GROUP[:NTR]))
TE_IDX = np.arange(NTR, len(full))
tick("setup neural selesai")


## Cross Encoder GRU400 train only

Varian dengan isi lebih panjang yaitu 400 token. Diberi bobot acak dan latih train only dengan satu seed untuk memberi keragaman pada ensemble akhir.


In [ ]:
cnt_v = Counter()
for s in COMP_T[:NTR] + COMP_C[:NTR]:
    cnt_v.update(norm(s).split())
VOCAB_G = {w: i + 2 for i, (w, c) in enumerate(cnt_v.most_common()) if c >= 2}
V_G = max(VOCAB_G.values()) + 1
MAXC_OLD = MAXC
MAXC = 400
ENC_G = encode(COMP_T, COMP_C, VOCAB_G)
MAXC = 200
tick(f"gru400 vocab {V_G} train saja MAXC 400")
GRU400_OOF = np.zeros(NTR)
GRU400_PTE = np.zeros(len(te))
for k, (i0, i1) in enumerate(FOLDS):
    ug = np.unique(GROUP[:NTR][i0])
    r0 = np.random.default_rng(1000 + k)
    vg = set(r0.choice(ug, size=max(int(0.15*len(ug)), 1), replace=False).tolist())
    iv = np.array([i for i in i0 if GROUP[i] in vg])
    it = np.array([i for i in i0 if GROUP[i] not in vg])
    Xiv = tuple(x[iv] for x in ENC_G)
    Tl, Cl, yl = [], [], []
    for i in it:
        Tl.append(COMP_T[i]); Cl.append(COMP_C[i]); yl.append(int(y[i]))
        if y[i] == 0:
            for _ in range(2):
                Tl.append(COMP_T[i]); Cl.append(COMP_C[i]); yl.append(0)
    pos0 = [int(i) for i in it if y[i] == 1]
    r1 = np.random.default_rng(100 * k)
    for i in r1.choice(pos0, size=int(1.5*len(pos0)), replace=True):
        i = int(i)
        t_, _ = CORR.corrupt(RAW_T[i], RAW_C[i], i)
        if t_ is None: continue
        Tl.append(prep_txt(t_)); Cl.append(COMP_C[i]); yl.append(0)
    yl = np.array(yl, dtype=np.float32)
    torch.manual_seed(SEED + 0)
    m = Enc(V_G).to(DEV)
    nn_train(m, encode(Tl, Cl, VOCAB_G), yl, Xiv, y[iv], 8, 2e-3, SEED*10)
    GRU400_OOF[i1] += nn_predict(m, *tuple(x[i1] for x in ENC_G))
    GRU400_PTE += nn_predict(m, *tuple(x[TE_IDX] for x in ENC_G)) / len(FOLDS)
    del m
    if DEV == "cuda": torch.cuda.empty_cache()
    tick(f"gru400 fold {k} foldval AUC {roc_auc_score(y[i1], GRU400_OOF[i1]):.4f}")
print(f"GRU400 OOF AUC {roc_auc_score(y, GRU400_OOF):.4f}")
MAXC = MAXC_OLD


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Training Retraining dan Submission <a name="8"></a>

Tahap ini merakit seluruh fitur menjadi matriks final lalu melatih ensemble GBDT dengan validasi StratifiedGroupKFold dan menentukan threshold dari OOF.


## Persiapan Data dan Threshold Tuning

Kita menggabungkan semua blok fitur yang telah dihitung train only dengan skor neural. Threshold dipilih dengan memaksimalkan Macro F1 pada OOF di ruang skor mentah maupun rank persentil.


In [ ]:
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import re as _re2
_WS_RE2 = _re2.compile("|".join(_re2.escape(c) for c in ["  ", "\u00a0", "\n", "\t", "\r", "\u2028", "\u2029", "\x0b", "\x0c"]))
def _fw(s):
    s = str(s)
    return bool(_WS_RE2.search(s)), len(_WS_RE2.findall(s))

WS4C = pd.DataFrame({
    "ws_title": [int(_fw(x)[0]) for x in RAW_T],
    "ws_content": [int(_fw(x)[0]) for x in RAW_C],
    "ws_n_title": [_fw(x)[1] for x in RAW_T],
    "ws_n_content": [_fw(x)[1] for x in RAW_C]})
print("flag whitespace per baris dihitung tanpa fitting label hanya turunan teks")

# rakit fitur utama
X_base = pd.concat([F_SEM, F_ALG, F_RET, F_DUP, F_NDP, F_EXT, WS4C], axis=1)
X_base["nn_gru400"] = np.r_[GRU400_OOF, GRU400_PTE]

COLS_BASE = SEM_COLS + ALG_COLS + RET_COLS + DUP_COLS + NDP_COLS + EXT_COLS + list(WS4C.columns) + ["nn_gru400"]
COLS_BASE = list(dict.fromkeys(COLS_BASE))

A = X_base.iloc[:NTR][COLS_BASE].astype("float32").reset_index(drop=True)
B = X_base.iloc[NTR:][COLS_BASE].astype("float32").reset_index(drop=True)
gtr = GROUP[:NTR]

print(f"matriks final train {A.shape} test {B.shape} fitur {len(COLS_BASE)}")

def tune_thr(yy, p):
    best = (-1.0, 0.5)
    for t in np.unique(np.quantile(p, np.linspace(0.0005, 0.75, 900))):
        s = f1_score(yy, (p >= t).astype(int), average="macro")
        if s > best[0]: best = (s, float(t))
    return best

# simpan untuk audit
A.to_csv(ARTIFACT_DIR / "features_train_preview.csv", index=False)
print("preview fitur disimpan ke artifact")


## Audit Split Internal

Sel ini menampilkan jumlah baris pada train val full train dan test agar proses seleksi dapat ditelusuri.


In [ ]:
split_audit = pd.DataFrame({
    "subset": ["train full", "test", "fitur train preview"],
    "rows": [len(A), len(B), len(COLS_BASE)],
    "keterangan": ["14397 baris label tersedia", "3603 baris untuk submission", "jumlah kolom fitur"]
})
display(split_audit)

# contoh fold
for k, (i0,i1) in enumerate(FOLDS):
    print(f"fold {k} train {len(i0)} val {len(i1)} val pos {(y[i1]==1).mean():.3f}")


## Model Final Ensemble

LightGBM XGBoost dan CatBoost masing masing dua seed sehingga enam anggota ensemble. Prediksi digabung dengan rata rata. Threshold diambil dari OOF yang memaksimalkan Macro F1 lalu diterapkan ke test di ruang skor yang sama.


In [ ]:
oof_r = np.zeros(NTR)
te_r = np.zeros(len(te))
nmod = 0
oof_per_model = {}
for seed in MODEL_SEEDS:
    for mn in ("lgb", "xgb", "cat"):
        o = np.zeros(NTR)
        p_te = np.zeros(len(te))
        skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42+seed)
        for k, (i0, i1) in enumerate(skf.split(A, y, gtr)):
            rs = seed*100 + k
            if mn == "lgb":
                m = lgb.LGBMClassifier(n_estimators=6000, learning_rate=0.02, num_leaves=63, colsample_bytree=0.6, subsample=0.85, subsample_freq=1, min_child_samples=25, reg_lambda=2.0, verbose=-1, random_state=rs, n_jobs=N_THREADS, deterministic=True, force_row_wise=True)
                m.fit(A.iloc[i0], y[i0], eval_set=[(A.iloc[i1], y[i1])], eval_metric="auc", callbacks=[lgb.early_stopping(250, verbose=False)])
            elif mn == "xgb":
                m = XGBClassifier(n_estimators=6000, learning_rate=0.02, max_depth=7, colsample_bytree=0.6, subsample=0.85, reg_lambda=2.0, eval_metric="auc", early_stopping_rounds=250, random_state=rs, n_jobs=N_THREADS, tree_method="hist")
                m.fit(A.iloc[i0], y[i0], eval_set=[(A.iloc[i1], y[i1])], verbose=False)
            else:
                m = CatBoostClassifier(iterations=6000, learning_rate=0.03, depth=7, l2_leaf_reg=4.0, eval_metric="AUC", random_seed=rs, verbose=0, early_stopping_rounds=250, thread_count=N_THREADS)
                m.fit(A.iloc[i0], y[i0], eval_set=(A.iloc[i1], y[i1]))
            o[i1] = m.predict_proba(A.iloc[i1])[:, 1]
            p_te += m.predict_proba(B)[:, 1] / N_SPLITS
        auc = roc_auc_score(y, o)
        macro, thr = tune_thr(y, o)
        print(f"{mn} seed {seed} AUC {auc:.4f} macroF1 {macro:.4f} thr {thr:.4f}")
        oof_per_model[f"{mn}_s{seed}"] = (auc, macro, thr)
        oof_r += o
        te_r += p_te
        nmod += 1

OOF = oof_r / nmod
PTE = te_r / nmod
AUC = roc_auc_score(y, OOF)
MACRO, THR = tune_thr(y, OOF)
pred_tr = (OOF >= THR).astype(int)
print("="*60)
print(f"ENSEMBLE OOF AUC {AUC:.4f} macroF1 {MACRO:.4f} thr {THR:.4f}")
print(f"F1 kelas 0 {f1_score(y, pred_tr, pos_label=0):.4f} F1 kelas 1 {f1_score(y, pred_tr, pos_label=1):.4f}")
free = F_DUP["dup_content_n"].to_numpy()[:NTR] == 0
print(f"bebas kembaran AUC {roc_auc_score(y[free], OOF[free]):.4f} n {int(free.sum())}")
print(f"berkembaran AUC {roc_auc_score(y[~free], OOF[~free]):.4f}")
print("="*60)


## Ringkasan Model

Sel ini merangkum performa tiap anggota dan ensemble akhir untuk keperluan audit.


In [ ]:
import pandas as pd
rows = []
for k, (auc, macro, thr) in oof_per_model.items():
    rows.append(dict(model=k, auc=auc, macroF1=macro, thr=thr))
rows.append(dict(model="ensemble_mean", auc=AUC, macroF1=MACRO, thr=THR))
summary = pd.DataFrame(rows).sort_values("macroF1", ascending=False)
display(summary)
summary.to_csv(ARTIFACT_DIR / "model_summary.csv", index=False)
print("ringkasan disimpan ke artifact/model_summary.csv")


## Full Retrain dan Pembuatan Submission

Threshold yang sudah dipilih dari OOF diterapkan ke skor test. Hasil akhir ditulis ke outputs/submission.csv dengan format id label.


In [ ]:
# submission final mengikuti sample submission yaitu id string teXXXXX
sub = pd.DataFrame({"id": te["id"].to_numpy(), "label": (PTE >= THR).astype(int)})
assert list(sub.columns) == ["id","label"]
assert len(sub) == 3603
assert sub["id"].is_unique
assert set(sub["label"].unique()) <= {0,1}
# pastikan urutan sama dengan test
assert (sub["id"].to_numpy() == te["id"].to_numpy()).all()

out_path = OUT_DIR / "submission.csv"
sub.to_csv(out_path, index=False)
print(f"submission ditulis ke {out_path}")
print(f"negatif diprediksi {int((sub.label==0).sum())} proporsi {(sub.label==0).mean():.2%} train {(y==0).mean():.2%}")
display(sub.head(8))
sub.describe()

# simpan metrics
metrics = dict(auc=float(AUC), macroF1=float(MACRO), thr=float(THR), n_train=int(NTR), n_test=len(te))
with open(ARTIFACT_DIR / "final_metrics.json","w",encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print(metrics)


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Reproducibility dan Audit Akhir <a name="9"></a>

Di akhir notebook submission hasil run ini dibandingkan dengan ekspektasi format dan metrik disimpan agar dapat direplikasi.


## Audit Kesamaan dan Format

Sel ini memeriksa bahwa submission akhir memiliki format yang sama dengan sample_submission.csv dan artifact final tersimpan.


In [ ]:
final_metrics_path = ARTIFACT_DIR / "final_metrics.json"
with final_metrics_path.open("r", encoding="utf-8") as f:
    final_metrics_loaded = json.load(f)
print("final metrics")
print(json.dumps(final_metrics_loaded, indent=2))

# cek format submission
sub_check = pd.read_csv(OUT_DIR / "submission.csv")
sample_check = pd.read_csv(DATA_DIR / "sample_submission.csv") if (DATA_DIR / "sample_submission.csv").exists() else None
print(f"submission rows {len(sub_check)} columns {list(sub_check.columns)}")
print(f"unique id {sub_check['id'].is_unique} label unik {sorted(sub_check['label'].unique())}")
if sample_check is not None:
    print(f"sample rows {len(sample_check)}")
    # cek prefix id
    print("contoh id submission", sub_check["id"].head(3).tolist())
    print("contoh id sample", sample_check["id"].head(3).tolist())

# simpan hash sederhana untuk audit
import hashlib
h = hashlib.md5(sub_check.to_csv(index=False).encode()).hexdigest()[:12]
print(f"md5 prefix submission {h}")
with open(ARTIFACT_DIR / "submission_hash.txt","w") as f:
    f.write(h+"\n")
print("audit selesai")


## Shortcut Inference dari Checkpoint Final

Bagian ini memberi jalan pintas bagi panitia untuk melakukan inferensi tanpa mengulang training yang panjang. Cukup muat artifact dan jalankan inferensi pada test baru bila dataset final dirilis.


In [ ]:
# contoh shortcut inference bila checkpoint tersedia
# pada run ini checkpoint berupa fitur dan model GBDT yang dilatih ulang
# untuk demonstrasi kita tunjukkan cara memuat kembali OOF dan threshold
print("shortcut inference memakai threshold final yang sudah disimpan")
print(f"gunakan threshold {final_metrics_loaded['thr']:.4f} pada skor rata rata ensemble")
print("langkahnya adalah hitung fitur yang sama untuk data baru lalu pakai model yang sama dan bandingkan dengan threshold tersebut")
# placeholder untuk data final bila ada
# final_test = pd.read_csv(DATA_DIR / "test_final.csv")
# ... hitung fitur dan prediksi ...
print("shortcut siap dipakai pada dataset final")


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Post Insight Analysis <a name="10"></a>

Analisis pasca training untuk memahami di mana model kuat dan di mana masih terbatas. Analisis ini memakai OOF sehingga tidak membutuhkan label test.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

pred_oof = (OOF >= THR_FINAL).astype(int)
print(classification_report(y, pred_oof, digits=4, target_names=["Tidak Sesuai 0","Sesuai 1"]))

cm = confusion_matrix(y, pred_oof, labels=[0,1])
fig, ax = plt.subplots(figsize=(4,3.2))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=["pred 0","pred 1"], yticklabels=["true 0","true 1"])
ax.set_title(f"Confusion Matrix OOF MacroF1 {MACRO:.3f}")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "confusion_oof.png", dpi=150)
plt.show()

# analisis per subset hard
hard_neg_mask = train_eda["overlap"].to_numpy() > 0.7
hard_pos_mask = train_eda["overlap"].to_numpy() < 0.5
# hanya pada label 0 dan 1 yang relevan
mask0 = y==0
mask1 = y==1
print("\nhard negative pada label 0")
if hard_neg_mask[mask0].sum()>0:
    print(f"jumlah {hard_neg_mask[mask0].sum()} macroF1 subset {f1_score(y[mask0][hard_neg_mask[mask0]], pred_oof[mask0][hard_neg_mask[mask0]], average='macro'):.4f}" if hard_neg_mask[mask0].sum()>1 else "terlalu sedikit")
print("hard positive pada label 1")
if hard_pos_mask[mask1].sum()>0:
    print(f"jumlah {hard_pos_mask[mask1].sum()} macroF1 subset {f1_score(y[mask1][hard_pos_mask[mask1]], pred_oof[mask1][hard_pos_mask[mask1]], average='macro'):.4f}" if hard_pos_mask[mask1].sum()>1 else "terlalu sedikit")

# distribusi skor OOF per label
fig, ax = plt.subplots(figsize=(6,3.2))
sns.kdeplot(OOF[y==1], fill=True, alpha=0.25, color="#22c55e", label="Sesuai 1")
sns.kdeplot(OOF[y==0], fill=True, alpha=0.25, color="#ef4444", label="Tidak Sesuai 0")
ax.axvline(THR_FINAL, color="black", ls="--", lw=1)
ax.text(THR_FINAL+0.01, ax.get_ylim()[1]*0.8, f"thr {THR_FINAL:.3f}", fontsize=8)
ax.set_title("Distribusi Skor OOF per Label")
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "score_dist_oof.png", dpi=150)
plt.show()


Insight kualitas data dan dampaknya
Distribusi kelas yang timpang dan overlap yang tinggi pada hard negative membuat model harus bekerja ekstra pada kelas kecil. Dampaknya sudah diantisipasi dengan pembobotan dan threshold yang memihak Macro F1.

Insight fitur dan dampaknya
Blok ALI dan perbandingan berpasangan memberi sinyal paling kuat ketika overlap gagal. Jendela selaras menangkap apakah inti judul didukung pada satu jendela isi sedangkan RET dan NDP menangkap apakah ada judul lain yang lebih berhak atas isi tersebut.

Keterbatasan dan saran
Model masih terbatas pada fitur berbasis token dan TF IDF plus cross encoder sederhana. Pengembangan lanjutan dapat mempertimbangkan augmentasi hard negative yang lebih variatif dan kalibrasi yang lebih halus per subset seperti hard negative versus bebas kembaran. Untuk deployment yang ringan model GBDT saja sudah memberi kualitas yang baik sedangkan cross encoder memberi tambahan pada kasus urutan yang ditukar.

Catatan akhir
Notebook ini patuh terhadap aturan from scratch dan train only. Seluruh indeks TF IDF Word2Vec dan sumber daya korpus di fit hanya pada train. Test hanya dipakai sebagai query.
